# 📊 SFC — Análisis de Performance: Visa vs Mastercard en Colombia

**Fuente:** Superintendencia Financiera de Colombia (datos.gov.co)  
**Cobertura:** Enero 2015 — dato más reciente disponible  
**Franquicias analizadas:** Visa, Mastercard (+ American Express, Diners, Tarjeta Débito como contexto)

---

### Estructura del notebook
1. **Carga y limpieza base** — pipeline unificado con conversión numérica
2. **Tablas temáticas** — Tarjetas, Transacciones, Cartera, Saldos
3. **Tabla maestra** — merge de todas las fuentes
4. **KPIs calculados** — market share, utilización, ticket, mora, churn, internacionalización
5. **Visualizaciones interactivas** — 10+ gráficas con Plotly
6. **Export** — CSVs individuales + master

## 0. Setup e Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Paleta de colores corporativa
COLORS = {
    'VISA':             '#1A1F71',   # azul oscuro Visa
    'MASTERCARD':       '#EB001B',   # rojo Mastercard
    'AMERICAN EXPRESS': '#007BC1',   # azul Amex
    'DINERS':           '#004B87',   # azul marino Diners
    'TARJETA DEBITO':   '#6B7280',   # gris
    'OTRAS TARJETAS DE CREDITO': '#9CA3AF',
}

PLOTLY_TEMPLATE = 'plotly_white'

print('✅ Setup completo')

## 1. Carga y Limpieza Base

In [ ]:
# ── 1.1 Descarga ──────────────────────────────────────────────────────────────
url = "https://www.datos.gov.co/api/v3/views/h2jg-r3zg/export.csv?accessType=DOWNLOAD&app_token=bHWsGtRFRP9x8Hl8lYivqM1hQ"

print('⏳ Descargando datos de la SFC...')
df_raw = pd.read_csv(url)
print(f'✅ {len(df_raw):,} registros descargados')
print(f'   Columnas: {list(df_raw.columns)}')

In [ ]:
# ── 1.2 Limpieza y normalización ──────────────────────────────────────────────
df = df_raw.copy()

# Renombrar columnas
df = df.drop(columns=['COD_UCA','TIPOENTIDAD','CODIGOENTIDAD','SUBCUENTA',
                       'PERSONA_NATURAL'], errors='ignore')
# Conservar PERSONA_JURIDICA para el filtro del dashboard
if 'PERSONA_JURIDICA' in df.columns:
    df['PERSONA_JURIDICA'] = (
        df['PERSONA_JURIDICA'].astype(str).str.replace(',','',regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )
df = df.rename(columns={
    'NOMBREENTIDAD': 'ENTIDAD',
    'FECHACORTE':    'MES',
    'NOMBRE_UCA':    'FRANQUICIA',
    'TOTAL_TARJETAS':'indicador'
})

# Unificar Credibanco-Visa → VISA
df['FRANQUICIA'] = df['FRANQUICIA'].replace('CREDIBANCO-VISA', 'VISA')

# Limpiar ENTIDADs con comillas literales desde la SFC
# (ej. '"RAPPIPAY"' → 'RAPPIPAY', '"BOLD C.F.", ...' → 'BOLD')
df['ENTIDAD'] = df['ENTIDAD'].str.strip('"').str.strip("'")
df['ENTIDAD'] = df['ENTIDAD'].str.replace(r'^BOLD.*', 'BOLD', regex=True)

# Excluir administradoras de bajo valor
df = df[df['FRANQUICIA'] != 'ADMINISTRADORAS DE SISTEMAS DE PAGO DE BAJO VALOR']

# ── CONVERSIÓN NUMÉRICA CENTRALIZADA ─────────────────────────────────────────
# Hacerla aquí evita repetirla en cada bloque posterior
df['indicador'] = (
    df['indicador']
    .astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

# Fecha
df['MES'] = pd.to_datetime(df['MES'], format='%d/%m/%Y', dayfirst=True)

print(f'✅ Datos limpios: {len(df):,} registros')
print(f'   Período: {df["MES"].min().strftime("%b %Y")} → {df["MES"].max().strftime("%b %Y")}')
print(f'   Entidades únicas: {df["ENTIDAD"].nunique()}')
print(f'   Franquicias: {sorted(df["FRANQUICIA"].unique())}')
print(f'   Indicadores únicos: {df["DESCRIPCION"].nunique()}')
display(df.head(3))

## 2. Tablas Temáticas

In [ ]:
# ── Helper: pivotear con nombres cortos ───────────────────────────────────────
def build_table(df_base, mapa, aggfunc='sum'):
    """
    Filtra df_base por las descripciones en `mapa`, renombra y pivotea.
    Retorna df ordenado por MES.
    """
    df_f = df_base[df_base['DESCRIPCION'].isin(mapa.keys())].copy()
    df_f['DESCRIPCION'] = df_f['DESCRIPCION'].map(mapa)
    df_p = df_f.pivot_table(
        index=['MES', 'ENTIDAD', 'FRANQUICIA'],
        columns='DESCRIPCION',
        values='indicador',
        aggfunc=aggfunc
    ).reset_index()
    return df_p.sort_values('MES').reset_index(drop=True)

print('✅ Helper definido')

In [ ]:
# ── 2.1 Tarjetas ──────────────────────────────────────────────────────────────
mapa_tarjetas = {
    'Número total de tarjetas débito  vigentes  a la fecha de corte':               'VIGENTES_FECHA_CORTE',
    'Número total de tarjetas de crédito vigentes  a la fecha de corte':            'VIGENTES_FECHA_CORTE',
    'Número total de tarjetas de crédito que cuentan con la tecnología "contactless" vigentes  a la fecha de corte': 'VIGENTES_FECHA_CORTE',
    'Número total de tarjetas débito que cuentan con la tecnología "contactless" vigentes  a la fecha de corte':     'VIGENTES_FECHA_CORTE',
    'Número total de tarjetas débito que no tienen chip de seguridad vigentes  a la fecha de corte':                 'VIGENTES_FECHA_CORTE',
    'Número total de tarjetas de crédito que no tienen chip de seguridad vigentes  a la fecha de corte':             'VIGENTES_FECHA_CORTE',
    'Número total de tarjetas de crédito vigentes durante el mes':  'VIGENTES_MES',
    'Número total de tarjetas débito vigentes durante el mes':      'VIGENTES_MES',
    'Número total de tarjetas de crédito canceladas':               'CANCELADAS',
    'Número total de tarjetas débito canceladas':                   'CANCELADAS',
    'Número total de tarjetas de créditos bloqueadas  temporalmente': 'BLOQUEADAS',
    'Número total de tarjetas débito bloqueadas temporalmente':       'BLOQUEADAS',
}

df_tarjetas = build_table(df, mapa_tarjetas, aggfunc='max')
print(f'✅ df_tarjetas: {df_tarjetas.shape}')
display(df_tarjetas.head(3))

In [ ]:
# ── 2.2 Compras y Avances (crédito) ───────────────────────────────────────────
mapa_tx = {
    'Número de transacciones por compras a nivel nacional con tarjeta de crédito':    'NUM_COMPRAS_NAL',
    'Monto de las transacciones por compras con tarjeta de crédito a nivel nacional': 'MTO_COMPRAS_NAL',
    'Número de  transacciones por avances a nivel nacional con tarjeta de crédito':   'NUM_AVANCES_NAL',
    'Monto de las transacciones por avances con tarjeta de crédito a nivel nacional': 'MTO_AVANCES_NAL',
    'Número de transacciones por compras en el exterior con tarjeta de crédito':      'NUM_COMPRAS_EXT',
    'Monto de las transacciones por compras en el exterior con tarjeta de crédito':   'MTO_COMPRAS_EXT',
    'Número de  transacciones por avances en el exterior con tarjeta de crédito':     'NUM_AVANCES_EXT',
    'Monto de las transacciones por avances en el exterior con tarjeta de crédito':   'MTO_AVANCES_EXT',
}

df_tx = build_table(df, mapa_tx, aggfunc='sum')
print(f'✅ df_tx: {df_tx.shape}')
display(df_tx.head(3))

In [ ]:
# ── 2.3 Compras y Retiros Débito ──────────────────────────────────────────────
mapa_debito = {
    'Número de transacciones por compras con tarjetas débito':  'NUM_COMPRAS_DEBITO',
    'Monto de transacciones por compras con tarjetas débito':   'MTO_COMPRAS_DEBITO',
    'Número de transacciones por retiros con tarjetas débito':  'NUM_RETIROS_DEBITO',
    'Monto de transacciones por retiros con tarjetas débito':   'MTO_RETIROS_DEBITO',
}

df_debito = build_table(df, mapa_debito, aggfunc='sum')
print(f'✅ df_debito: {df_debito.shape}')
display(df_debito.head(3))

In [ ]:
# ── 2.4 Cartera: Intereses y Castigos ─────────────────────────────────────────
mapa_cartera = {
    'Monto de los intereses corrientes por compras y avances con tarjeta de crédito':         'INTERESES_CORRIENTES',
    'Monto de los intereses de mora por compras y avances  con tarjeta de crédito':           'INTERESES_MORA',
    'Monto de los castigos de cartera por tarjeta de crédito, únicamente capital.':           'CASTIGOS_CAPITAL',
    'Monto de los castigos de cartera por tarjeta de crédito, conceptos diferentes a capital':'CASTIGOS_OTROS',
}

df_cartera = build_table(df, mapa_cartera, aggfunc='sum')
print(f'✅ df_cartera: {df_cartera.shape}')
display(df_cartera.head(3))

In [ ]:
# ── 2.5 Saldos y Cupos ────────────────────────────────────────────────────────
mapa_saldos = {
    'Saldo de la cartera por tarjeta de crédito':                               'SALDO_CARTERA',
    'Total cupo de crédito no utilizado por todos los tarjetahabientes':        'CUPO_NO_UTILIZADO',
}

df_saldos = build_table(df, mapa_saldos, aggfunc='sum')
print(f'✅ df_saldos: {df_saldos.shape}')
display(df_saldos.head(3))

In [ ]:
# ── 2.6 TII (Tarifa Interbancaria de Intercambio) ─────────────────────────────
mapa_tii = {
    'INGRESOS POR TARIFA INTERBANCARIA DE INT':                                              'TII_INGRESOS',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Master Débito':      'TII_INGRESOS',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Electrón':    'TII_INGRESOS',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Visa':        'TII_INGRESOS',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Maestro':     'TII_INGRESOS',
    'Ingresos Comisión de Adquirencia por Tarjeta Débito':                                   'COMISION_ADQUIRENCIA',
    'Ingresos Comisión de Adquirencia por Tarjeta de Crédito':                               'COMISION_ADQUIRENCIA',
    'Gastos por Tarifa Interbancaria de Intercambio - TII por Tarjeta de Crédito':           'TII_GASTOS',
    'Gastos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Maestro':       'TII_GASTOS',
    'Gastos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Visa':          'TII_GASTOS',
    'Gastos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Electrón':      'TII_GASTOS',
    'Gastos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Master Débito':        'TII_GASTOS',
    'GASTOS POR TARIFA INTER.':                                                              'TII_GASTOS',
}

df_tii = build_table(df, mapa_tii, aggfunc='sum')
print(f'✅ df_tii: {df_tii.shape}')
display(df_tii.head(3))

## 3. Tabla Maestra

In [ ]:
# ── 3.1 Merge de todas las fuentes ────────────────────────────────────────────
KEY = ['MES', 'ENTIDAD', 'FRANQUICIA']

df_master = (
    df_tarjetas
    .merge(df_tx,      on=KEY, how='outer')
    .merge(df_debito,  on=KEY, how='outer')
    .merge(df_cartera, on=KEY, how='outer')
    .merge(df_saldos,  on=KEY, how='outer')
    .merge(df_tii,     on=KEY, how='outer')
    .sort_values('MES')
    .reset_index(drop=True)
)

print(f'✅ df_master: {df_master.shape}')
print(f'   Columnas: {list(df_master.columns)}')
display(df_master.head(5))

## 4. KPIs Calculados

In [ ]:
# ── 4.1 Agregar KPIs en df_master ─────────────────────────────────────────────

# Ticket promedio por compra (crédito nacional)
df_master['TICKET_PROMEDIO'] = (
    df_master['MTO_COMPRAS_NAL'] / df_master['NUM_COMPRAS_NAL'].replace(0, np.nan)
)

# Utilización de cupo (saldo / (saldo + cupo disponible))
df_master['UTILIZACION_CUPO'] = (
    df_master['SALDO_CARTERA'] /
    (df_master['SALDO_CARTERA'] + df_master['CUPO_NO_UTILIZADO']).replace(0, np.nan)
)

# Ratio de mora (intereses mora / intereses corrientes)
df_master['RATIO_MORA'] = (
    df_master['INTERESES_MORA'] / df_master['INTERESES_CORRIENTES'].replace(0, np.nan)
)

# Churn rate implícito (canceladas / vigentes mes)
df_master['CHURN_RATE'] = (
    df_master['CANCELADAS'] / df_master['VIGENTES_MES'].replace(0, np.nan)
)

# % compras internacionales sobre total
df_master['PCT_COMPRAS_EXT'] = (
    df_master['MTO_COMPRAS_EXT'] /
    (df_master['MTO_COMPRAS_NAL'] + df_master['MTO_COMPRAS_EXT']).replace(0, np.nan)
)

# Castigo total (capital + otros)
df_master['CASTIGOS_TOTAL'] = (
    df_master['CASTIGOS_CAPITAL'].fillna(0) + df_master['CASTIGOS_OTROS'].fillna(0)
)

# Ratio castigos / saldo cartera (pérdida esperada)
df_master['TASA_CASTIGO'] = (
    df_master['CASTIGOS_TOTAL'] / df_master['SALDO_CARTERA'].replace(0, np.nan)
)

# Volumen total transaccional (compras + avances, nal + ext)
for col in ['MTO_COMPRAS_NAL','MTO_AVANCES_NAL','MTO_COMPRAS_EXT','MTO_AVANCES_EXT']:
    df_master[col] = df_master[col].fillna(0)

df_master['VOLUMEN_TX_TOTAL'] = (
    df_master['MTO_COMPRAS_NAL'] + df_master['MTO_AVANCES_NAL'] +
    df_master['MTO_COMPRAS_EXT'] + df_master['MTO_AVANCES_EXT']
)

# TII neto (ingresos - gastos)
df_master['TII_NETO'] = (
    df_master['TII_INGRESOS'].fillna(0) - df_master['TII_GASTOS'].fillna(0)
)

print('✅ KPIs calculados en df_master')
kpi_cols = ['TICKET_PROMEDIO','UTILIZACION_CUPO','RATIO_MORA','CHURN_RATE',
            'PCT_COMPRAS_EXT','TASA_CASTIGO','VOLUMEN_TX_TOTAL','TII_NETO']
display(df_master[['MES','ENTIDAD','FRANQUICIA'] + kpi_cols].dropna(subset=['TICKET_PROMEDIO']).head(5))

In [ ]:
# ── 4.2 Vista mensual agregada por FRANQUICIA (para gráficas) ─────────────────

# Solo Visa y Mastercard (crédito) para la mayoría de análisis
CRED_FRANQ = ['VISA', 'MASTERCARD', 'AMERICAN EXPRESS', 'DINERS']

agg_cols = {
    'VIGENTES_FECHA_CORTE': 'sum',
    'VIGENTES_MES':         'sum',
    'CANCELADAS':           'sum',
    'BLOQUEADAS':           'sum',
    'NUM_COMPRAS_NAL':      'sum',
    'MTO_COMPRAS_NAL':      'sum',
    'NUM_AVANCES_NAL':      'sum',
    'MTO_AVANCES_NAL':      'sum',
    'NUM_COMPRAS_EXT':      'sum',
    'MTO_COMPRAS_EXT':      'sum',
    'MTO_AVANCES_EXT':      'sum',
    'INTERESES_CORRIENTES': 'sum',
    'INTERESES_MORA':       'sum',
    'CASTIGOS_CAPITAL':     'sum',
    'CASTIGOS_OTROS':       'sum',
    'CASTIGOS_TOTAL':       'sum',
    'SALDO_CARTERA':        'sum',
    'CUPO_NO_UTILIZADO':    'sum',
    'VOLUMEN_TX_TOTAL':     'sum',
    'TII_INGRESOS':         'sum',
    'TII_GASTOS':           'sum',
    'TII_NETO':             'sum',
}

# Agregado crédito por franquicia
df_agg = (
    df_master[df_master['FRANQUICIA'].isin(CRED_FRANQ)]
    .groupby(['MES', 'FRANQUICIA'])
    .agg({k: v for k, v in agg_cols.items() if k in df_master.columns})
    .reset_index()
)

# Recalcular KPIs sobre el agregado
df_agg['TICKET_PROMEDIO']   = df_agg['MTO_COMPRAS_NAL'] / df_agg['NUM_COMPRAS_NAL'].replace(0, np.nan)
df_agg['UTILIZACION_CUPO']  = df_agg['SALDO_CARTERA'] / (df_agg['SALDO_CARTERA'] + df_agg['CUPO_NO_UTILIZADO']).replace(0, np.nan)
df_agg['RATIO_MORA']        = df_agg['INTERESES_MORA'] / df_agg['INTERESES_CORRIENTES'].replace(0, np.nan)
df_agg['CHURN_RATE']        = df_agg['CANCELADAS'] / df_agg['VIGENTES_MES'].replace(0, np.nan)
df_agg['PCT_COMPRAS_EXT']   = df_agg['MTO_COMPRAS_EXT'] / (df_agg['MTO_COMPRAS_NAL'] + df_agg['MTO_COMPRAS_EXT']).replace(0, np.nan)
df_agg['TASA_CASTIGO']      = df_agg['CASTIGOS_TOTAL'] / df_agg['SALDO_CARTERA'].replace(0, np.nan)

# Market share por NÚMERO de tarjetas
total_tarjetas = df_agg.groupby('MES')['VIGENTES_FECHA_CORTE'].transform('sum')
df_agg['MS_TARJETAS'] = df_agg['VIGENTES_FECHA_CORTE'] / total_tarjetas

# Market share por VOLUMEN de transacciones
total_volumen = df_agg.groupby('MES')['VOLUMEN_TX_TOTAL'].transform('sum')
df_agg['MS_VOLUMEN'] = df_agg['VOLUMEN_TX_TOTAL'] / total_volumen

# Market share por SALDO de cartera
total_saldo = df_agg.groupby('MES')['SALDO_CARTERA'].transform('sum')
df_agg['MS_SALDO'] = df_agg['SALDO_CARTERA'] / total_saldo

# Market share por MONTO DE COMPRAS EN CRÉDITO (nal + ext) — el más relevante comercialmente
df_agg['MTO_COMPRAS_CREDITO'] = df_agg['MTO_COMPRAS_NAL'] + df_agg['MTO_COMPRAS_EXT']
total_compras_cred = df_agg.groupby('MES')['MTO_COMPRAS_CREDITO'].transform('sum')
df_agg['MS_COMPRAS_CREDITO'] = df_agg['MTO_COMPRAS_CREDITO'] / total_compras_cred

# Solo Visa y Mastercard
df_vm = df_agg[df_agg['FRANQUICIA'].isin(['VISA', 'MASTERCARD'])].copy()

print(f'✅ df_agg: {df_agg.shape} | df_vm: {df_vm.shape}')
display(df_agg.tail(8))

## 5. Visualizaciones

In [ ]:
# ── VIZ 1: Market Share — 4 dimensiones ───────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '🏆 Monto Compras Crédito (nal + ext)  ← principal',
        'N° Tarjetas Vigentes',
        'Volumen TX Total (compras + avances)',
        'Saldo de Cartera'
    ),
    shared_yaxes=False
)

panels = [
    ('MS_COMPRAS_CREDITO', 1, 1),
    ('MS_TARJETAS',        1, 2),
    ('MS_VOLUMEN',         2, 1),
    ('MS_SALDO',           2, 2),
]

for col, r, c in panels:
    for franq in ['VISA', 'MASTERCARD']:
        sub = df_vm[df_vm['FRANQUICIA'] == franq]
        fig.add_trace(
            go.Scatter(
                x=sub['MES'], y=sub[col],
                name=franq, legendgroup=franq,
                showlegend=(r == 1 and c == 1),
                line=dict(color=COLORS[franq], width=2.5 if col == 'MS_COMPRAS_CREDITO' else 2),
                hovertemplate=f'{franq}<br>%{{x|%b %Y}}: %{{y:.1%}}<extra></extra>'
            ),
            row=r, col=c
        )

fig.update_yaxes(tickformat='.0%')
fig.update_layout(
    title_text='<b>Market Share Visa vs Mastercard — Colombia (2015–hoy)</b>',
    template=PLOTLY_TEMPLATE, height=620, hovermode='x unified'
)
fig.show()

In [ ]:
# ── VIZ 2: Evolución de tarjetas vigentes (todas las franquicias) ──────────────
fig = px.area(
    df_agg,
    x='MES', y='VIGENTES_FECHA_CORTE',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>Tarjetas de Crédito Vigentes por Franquicia</b>',
    labels={'VIGENTES_FECHA_CORTE': 'Tarjetas Vigentes', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(hovermode='x unified', height=420)
fig.show()

In [ ]:
# ── VIZ 3: Volumen de compras nacionales ──────────────────────────────────────
fig = px.line(
    df_vm,
    x='MES', y='MTO_COMPRAS_NAL',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>Monto Compras Nacionales — Visa vs Mastercard (COP)</b>',
    labels={'MTO_COMPRAS_NAL': 'Monto COP', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(hovermode='x unified', height=420)
fig.update_yaxes(tickformat=',.0f')
fig.show()

In [ ]:
# ── VIZ 4: Ticket Promedio por compra ─────────────────────────────────────────
fig = px.line(
    df_vm,
    x='MES', y='TICKET_PROMEDIO',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>Ticket Promedio por Compra Nacional (COP)</b><br><sup>Proxy de perfil socioeconómico del tarjetahabiente</sup>',
    labels={'TICKET_PROMEDIO': 'COP por transacción', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(hovermode='x unified', height=420)
fig.update_yaxes(tickformat=',.0f')
fig.show()

In [ ]:
# ── VIZ 5: Panel de salud de cartera ──────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Saldo Cartera (COP)',
        'Utilización de Cupo (%)',
        'Ratio de Mora (mora / corrientes)',
        'Tasa de Castigo (castigos / saldo)'
    )
)

panels = [
    ('SALDO_CARTERA',    ',.0f',  1, 1),
    ('UTILIZACION_CUPO', '.1%',   1, 2),
    ('RATIO_MORA',       '.2%',   2, 1),
    ('TASA_CASTIGO',     '.3%',   2, 2),
]

for col, fmt, r, c in panels:
    for franq in ['VISA', 'MASTERCARD']:
        sub = df_vm[df_vm['FRANQUICIA'] == franq]
        fig.add_trace(
            go.Scatter(
                x=sub['MES'], y=sub[col],
                name=franq, legendgroup=franq,
                showlegend=(r == 1 and c == 1),
                line=dict(color=COLORS[franq], width=2),
                hovertemplate=f'{franq}: %{{y:{fmt}}}<extra></extra>'
            ),
            row=r, col=c
        )

fig.update_layout(
    title_text='<b>Panel de Salud de Cartera — Visa vs Mastercard</b>',
    template=PLOTLY_TEMPLATE, height=620, hovermode='x unified'
)
fig.show()

In [ ]:
# ── VIZ 6: Churn Rate ─────────────────────────────────────────────────────────
fig = px.line(
    df_vm,
    x='MES', y='CHURN_RATE',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>Churn Rate Implícito — Tarjetas Canceladas / Vigentes del Mes</b>',
    labels={'CHURN_RATE': 'Churn rate', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(hovermode='x unified', height=420)
fig.update_yaxes(tickformat='.2%')
fig.show()

In [ ]:
# ── VIZ 7: Internacionalización ───────────────────────────────────────────────
fig = px.line(
    df_vm,
    x='MES', y='PCT_COMPRAS_EXT',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>% Compras en el Exterior sobre Total</b><br><sup>Nota: caída 2020 refleja cierre de fronteras COVID</sup>',
    labels={'PCT_COMPRAS_EXT': '% compras exterior', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(hovermode='x unified', height=420)
fig.update_yaxes(tickformat='.1%')
fig.show()

In [ ]:
# ── VIZ 8: TII — Ingresos netos de intercambio ────────────────────────────────
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('TII Ingresos vs Gastos', 'TII Neto (Ingresos - Gastos)'))

for franq in ['VISA', 'MASTERCARD']:
    sub = df_vm[df_vm['FRANQUICIA'] == franq]
    fig.add_trace(
        go.Scatter(x=sub['MES'], y=sub['TII_INGRESOS'],
                   name=f'{franq} — Ingresos', legendgroup=franq,
                   line=dict(color=COLORS[franq], dash='solid', width=2),
                   hovertemplate='%{y:,.0f}<extra></extra>'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=sub['MES'], y=sub['TII_GASTOS'],
                   name=f'{franq} — Gastos', legendgroup=franq,
                   line=dict(color=COLORS[franq], dash='dot', width=1.5),
                   hovertemplate='%{y:,.0f}<extra></extra>'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=sub['MES'], y=sub['TII_NETO'],
                   name=franq, legendgroup=franq, showlegend=False,
                   line=dict(color=COLORS[franq], width=2),
                   fill='tozeroy', fillcolor=COLORS[franq],
                   opacity=0.2),
        row=1, col=2
    )

fig.update_layout(
    title_text='<b>Tarifa Interbancaria de Intercambio (TII)</b>',
    template=PLOTLY_TEMPLATE, height=420, hovermode='x unified'
)
fig.show()

In [ ]:
# ── VIZ 9: Composición del gasto por tipo (compras vs avances) ────────────────
df_tipo = df_vm.copy()
df_tipo['PCT_AVANCES'] = (
    df_tipo['MTO_AVANCES_NAL'] /
    (df_tipo['MTO_COMPRAS_NAL'] + df_tipo['MTO_AVANCES_NAL']).replace(0, np.nan)
)

fig = px.line(
    df_tipo,
    x='MES', y='PCT_AVANCES',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>% Avances sobre Total Crédito Nacional</b><br><sup>Mayor avance = mayor ingreso por intereses pero mayor riesgo</sup>',
    labels={'PCT_AVANCES': '% avances', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(hovermode='x unified', height=420)
fig.update_yaxes(tickformat='.1%')
fig.show()

In [ ]:
# ── VIZ 10: Top 10 entidades por volumen de compras (último mes disponible) ────
ultimo_mes = df_master['MES'].max()
df_top = (
    df_master[
        (df_master['MES'] == ultimo_mes) &
        (df_master['FRANQUICIA'].isin(['VISA', 'MASTERCARD']))
    ]
    .groupby(['ENTIDAD', 'FRANQUICIA'])['MTO_COMPRAS_NAL']
    .sum()
    .reset_index()
    .sort_values('MTO_COMPRAS_NAL', ascending=False)
    .head(20)
)

fig = px.bar(
    df_top,
    x='MTO_COMPRAS_NAL', y='ENTIDAD',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    orientation='h',
    title=f'<b>Top Entidades por Monto Compras Nacionales — {ultimo_mes.strftime("%b %Y")}</b>',
    labels={'MTO_COMPRAS_NAL': 'Monto COP', 'ENTIDAD': ''},
    template=PLOTLY_TEMPLATE,
    barmode='group'
)
fig.update_layout(height=550, yaxis_categoryorder='total ascending')
fig.update_xaxes(tickformat=',.0f')
fig.show()

In [ ]:
# ── VIZ 11: Snapshot de KPIs — último mes disponible ──────────────────────────
snap = df_agg[df_agg['MES'] == ultimo_mes][[
    'FRANQUICIA', 'MS_COMPRAS_CREDITO', 'MTO_COMPRAS_CREDITO',
    'MS_TARJETAS', 'MS_VOLUMEN', 'MS_SALDO',
    'TICKET_PROMEDIO', 'UTILIZACION_CUPO', 'RATIO_MORA',
    'CHURN_RATE', 'TASA_CASTIGO', 'PCT_COMPRAS_EXT'
]].copy()

print(f"\n📊 SNAPSHOT DE KPIs — {ultimo_mes.strftime('%B %Y').upper()}")
print('='*80)
for _, row in snap.iterrows():
    print(f"\n🏦 {row['FRANQUICIA']}")
    print(f"   🏆 MS Compras Crédito →  {row['MS_COMPRAS_CREDITO']:.1%}  ({row['MTO_COMPRAS_CREDITO']:>20,.0f} COP)")
    print(f"   Market Share  →  Tarjetas: {row['MS_TARJETAS']:.1%}  |  Volumen: {row['MS_VOLUMEN']:.1%}  |  Saldo: {row['MS_SALDO']:.1%}")
    print(f"   Ticket prom.  →  {row['TICKET_PROMEDIO']:>16,.0f} COP")
    print(f"   Utiliz. cupo  →  {row['UTILIZACION_CUPO']:.1%}")
    print(f"   Ratio mora    →  {row['RATIO_MORA']:.2%}")
    print(f"   Churn rate    →  {row['CHURN_RATE']:.2%}")
    print(f"   Tasa castigo  →  {row['TASA_CASTIGO']:.3%}")
    print(f"   % ext.        →  {row['PCT_COMPRAS_EXT']:.1%}")
print('='*80)

In [ ]:
# ── VIZ 12: Crecimiento YoY del volumen de transacciones ──────────────────────
df_yoy = df_vm.copy()
df_yoy = df_yoy.sort_values(['FRANQUICIA', 'MES'])
df_yoy['VOLUMEN_YOY'] = df_yoy.groupby('FRANQUICIA')['VOLUMEN_TX_TOTAL'].pct_change(12)
df_yoy = df_yoy.dropna(subset=['VOLUMEN_YOY'])

fig = px.line(
    df_yoy,
    x='MES', y='VOLUMEN_YOY',
    color='FRANQUICIA',
    color_discrete_map=COLORS,
    title='<b>Crecimiento YoY — Volumen de Transacciones</b>',
    labels={'VOLUMEN_YOY': 'Crecimiento anual', 'MES': ''},
    template=PLOTLY_TEMPLATE
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(hovermode='x unified', height=420)
fig.update_yaxes(tickformat='.0%')
fig.show()

## 6. Dashboard Ejecutivo (HTML standalone)

In [ ]:
# ── 6.1 Preparar datos para el dashboard ──────────────────────────────────────
import json

# Asegurar que MTO_COMPRAS_CREDITO existe en df_master
if 'MTO_COMPRAS_CREDITO' not in df_master.columns:
    df_master['MTO_COMPRAS_CREDITO'] = (
        df_master['MTO_COMPRAS_NAL'].fillna(0) +
        df_master['MTO_COMPRAS_EXT'].fillna(0)
    )

# Asegurar CASTIGOS_TOTAL
if 'CASTIGOS_TOTAL' not in df_master.columns:
    df_master['CASTIGOS_TOTAL'] = (
        df_master['CASTIGOS_CAPITAL'].fillna(0) +
        df_master['CASTIGOS_OTROS'].fillna(0)
    )

DASH_FRANQS = ['VISA', 'MASTERCARD', 'AMERICAN EXPRESS', 'DINERS', 'OTRAS TARJETAS DE CREDITO']

dash_cols_want = [
    'MES', 'ENTIDAD', 'FRANQUICIA',
    'MTO_COMPRAS_NAL', 'MTO_COMPRAS_EXT', 'MTO_COMPRAS_CREDITO',
    'NUM_COMPRAS_NAL', 'NUM_COMPRAS_EXT',
    'VIGENTES_FECHA_CORTE', 'CANCELADAS', 'VIGENTES_MES',
    'SALDO_CARTERA', 'CUPO_NO_UTILIZADO',
    'INTERESES_CORRIENTES', 'INTERESES_MORA',
    'CASTIGOS_CAPITAL', 'CASTIGOS_OTROS', 'CASTIGOS_TOTAL',
    'PERSONA_JURIDICA',
]

df_dash = (
    df_master[df_master['FRANQUICIA'].isin(DASH_FRANQS)]
    [[c for c in dash_cols_want if c in df_master.columns]]
    .copy()
)
df_dash['MES'] = df_dash['MES'].dt.strftime('%Y-%m')
df_dash = df_dash.where(pd.notna(df_dash), None)

# Incluir todas las entidades de crédito + débito en el dropdown
bancos_cred = set(df_dash['ENTIDAD'].dropna().unique().tolist())
# df_deb puede no estar definido aún si se corre solo celda 33 parcialmente
try:
    bancos_deb = set(df_deb['ENTIDAD'].dropna().unique().tolist())
except:
    bancos_deb = set()
bancos = sorted(bancos_cred | bancos_deb)
ultimo_mes_str = df_dash['MES'].max()
dash_data_json = df_dash.to_json(orient='records', force_ascii=False)
bancos_json    = json.dumps(bancos, ensure_ascii=False)

# Diagnóstico rápido
sample = df_dash[df_dash['FRANQUICIA']=='VISA'].sort_values('MES').tail(1)
print(f"✅ {len(df_dash):,} filas | {df_dash['MES'].min()} → {ultimo_mes_str}")
print(f"   Bancos: {len(bancos)}")
print(f"   Franquicias: {df_dash['FRANQUICIA'].unique().tolist()}")
print(f"\nMuestra Visa último mes:")
for col in ['MTO_COMPRAS_NAL','MTO_COMPRAS_EXT','MTO_COMPRAS_CREDITO','SALDO_CARTERA']:
    if col in sample.columns:
        val = sample[col].sum()
        print(f"   {col}: {val:,.0f}  ({val/1e12:.3f} T COP)" if val else f"   {col}: None/0")

# ── 6.1b Preparar datos de débito ────────────────────────────────────────────
# Metodología:
#   1. Monto de compras por emisor = volumen real (excluye redes de bajo valor)
#   2. TII por franquicia = llave de distribución Visa vs MC dentro de cada banco
#   3. Regla de tres: monto_visa = monto_banco × (tii_visa / tii_banco_total)

EXCLUIR_REDES = [
    'REDEBAN S.A.',
    'MASTERCARD COLOMBIA ADMINISTRADORA S.A.',
    'CREDIBANCO S.A. PUDIENDO SIN PERDER SU NATURALEZA UTILIZAR LA SIGLA CREDIBANCO',
    'VISA COLOMBIA SUPPORT SERVICES SOCIEDAD ANONIMA',
    'VISIONAMOS SISTEMA DE PAGO COOPERATIVO',
]

TII_DESCRIPS = [
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Visa',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Electrón',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Débito Maestro',
    'Ingresos por Tarifa Interbancaria de Intercambio - TII por Tarjeta Master Débito',
]

def franq_from_desc(desc):
    d = desc.upper()
    if 'ELECTR' in d or 'DÉBITO VISA' in d or 'DEBITO VISA' in d: return 'VISA'
    return 'MASTERCARD'

# ── Paso 1: monto de compras débito por banco/mes (solo emisores) ─────────────
df_monto_deb = df_raw[
    (df_raw['DESCRIPCION'] == 'Monto de transacciones por compras con tarjetas débito') &
    (~df_raw['NOMBREENTIDAD'].isin(EXCLUIR_REDES))
].copy()
df_monto_deb['MES'] = pd.to_datetime(df_monto_deb['FECHACORTE'], format='%d/%m/%Y',
                                      dayfirst=True, errors='coerce').dt.strftime('%Y-%m')
df_monto_deb['MONTO_TOTAL'] = pd.to_numeric(
    df_monto_deb['TOTAL_TARJETAS'].astype(str).str.replace(',','', regex=False),
    errors='coerce').fillna(0)
df_monto_deb = df_monto_deb.groupby(['MES','NOMBREENTIDAD'])['MONTO_TOTAL'].sum().reset_index()
df_monto_deb = df_monto_deb.rename(columns={'NOMBREENTIDAD':'ENTIDAD'})

# ── Paso 2: TII por banco/mes/franquicia ──────────────────────────────────────
df_tii = df_raw[df_raw['DESCRIPCION'].isin(TII_DESCRIPS)].copy()
df_tii['MES'] = pd.to_datetime(df_tii['FECHACORTE'], format='%d/%m/%Y',
                                dayfirst=True, errors='coerce').dt.strftime('%Y-%m')
df_tii['FRANQUICIA'] = df_tii['DESCRIPCION'].apply(franq_from_desc)
df_tii['TII'] = pd.to_numeric(
    df_tii['TOTAL_TARJETAS'].astype(str).str.replace(',','', regex=False),
    errors='coerce').fillna(0)
df_tii_grp = df_tii.groupby(['MES','NOMBREENTIDAD','FRANQUICIA'])['TII'].sum().reset_index()
df_tii_grp = df_tii_grp.rename(columns={'NOMBREENTIDAD':'ENTIDAD'})

# TII total por banco/mes
df_tii_tot = df_tii_grp.groupby(['MES','ENTIDAD'])['TII'].sum().reset_index()
df_tii_tot = df_tii_tot.rename(columns={'TII':'TII_TOTAL'})
df_tii_grp = df_tii_grp.merge(df_tii_tot, on=['MES','ENTIDAD'], how='left')
df_tii_grp['PROP'] = df_tii_grp['TII'] / df_tii_grp['TII_TOTAL'].replace(0, 1)

# ── Paso 3: regla de tres → monto por franquicia ─────────────────────────────
# Bancos 100% Mastercard conocidos (nunca o ya no reportan TII Visa)
BANCOS_100_MC = [
    'SCOTIABANK COLPATRIA S.A.',
    'BANCO FALABELLA S.A.',
    'MIBANCO S.A.',
]
# Para estos bancos construir filas MC directamente desde df_monto_deb
df_mc_default = df_monto_deb[
    df_monto_deb['ENTIDAD'].isin(BANCOS_100_MC)
].copy()
df_mc_default['FRANQUICIA'] = 'MASTERCARD'
df_mc_default['PROP']       = 1.0
df_mc_default['TII']        = 0
df_mc_default['TII_TOTAL']  = 0

# Para el resto: left merge con TII (pueden tener Visa/MC split)
df_resto_monto = df_monto_deb[~df_monto_deb['ENTIDAD'].isin(BANCOS_100_MC)].copy()
df_deb_split = df_resto_monto.merge(df_tii_grp, on=['MES','ENTIDAD'], how='left')
df_deb_split['FRANQUICIA'] = df_deb_split['FRANQUICIA'].fillna('MASTERCARD')
df_deb_split['PROP']       = df_deb_split['PROP'].fillna(1.0)
df_deb_split['TII']        = df_deb_split['TII'].fillna(0)
df_deb_split['TII_TOTAL']  = df_deb_split['TII_TOTAL'].fillna(0)

# Combinar
df_deb = pd.concat([df_deb_split, df_mc_default], ignore_index=True)
df_deb['MTO_COMPRAS_CREDITO'] = df_deb['PROP'] * df_deb['MONTO_TOTAL']

# ── Paso 4: agregar tarjetas vigentes y txn (proporcional por TII) ─────────────
MAPA_VIG_DEB = {
    'Número total de tarjetas débito  vigentes  a la fecha de corte': 'VIGENTES_FECHA_CORTE',
    'Número de transacciones por compras con tarjetas débito': 'NUM_COMPRAS_NAL',
}
df_vig = df_raw[df_raw['DESCRIPCION'].isin(MAPA_VIG_DEB.keys())].copy()
df_vig['MES'] = pd.to_datetime(df_vig['FECHACORTE'], format='%d/%m/%Y',
                                dayfirst=True, errors='coerce').dt.strftime('%Y-%m')
df_vig['DESC_MAP'] = df_vig['DESCRIPCION'].map(MAPA_VIG_DEB)
df_vig['val'] = pd.to_numeric(
    df_vig['TOTAL_TARJETAS'].astype(str).str.replace(',','', regex=False),
    errors='coerce').fillna(0)
df_vig_piv = df_vig.pivot_table(
    index=['MES','NOMBREENTIDAD'], columns='DESC_MAP', values='val', aggfunc='sum'
).reset_index().rename(columns={'NOMBREENTIDAD':'ENTIDAD'})
df_vig_piv.columns.name = None
for c in ['VIGENTES_FECHA_CORTE','NUM_COMPRAS_NAL']:
    if c not in df_vig_piv.columns: df_vig_piv[c] = 0

df_deb = df_deb.merge(df_vig_piv, on=['MES','ENTIDAD'], how='left')
# Distribute vigentes/txn proportionally
for col in ['VIGENTES_FECHA_CORTE','NUM_COMPRAS_NAL']:
    df_deb[col] = df_deb.get(col, pd.Series(0, index=df_deb.index)).fillna(0) * df_deb['PROP']

# ── Paso 5: Nequi — separar de Bancolombia ────────────────────────────────────
# Bancolombia Visa débito = Nequi (evidencia: TII Visa domina ~97% en Bancolombia)
banc_mask = df_deb['ENTIDAD'].str.contains('BANCOLOMBIA', na=False)
nequi_rows = df_deb[banc_mask & (df_deb['FRANQUICIA']=='VISA')].copy()
nequi_rows['ENTIDAD'] = 'NEQUI (Bancolombia)'
banc_mc_rows = df_deb[banc_mask & (df_deb['FRANQUICIA']=='MASTERCARD')].copy()
otros_rows   = df_deb[~banc_mask].copy()
df_deb = pd.concat([nequi_rows, banc_mc_rows, otros_rows], ignore_index=True)

# ── Finalizar columnas ────────────────────────────────────────────────────────
for col in ['MTO_COMPRAS_NAL','MTO_COMPRAS_EXT','NUM_COMPRAS_EXT','CANCELADAS',
            'VIGENTES_MES','SALDO_CARTERA','CUPO_NO_UTILIZADO','INTERESES_CORRIENTES',
            'INTERESES_MORA','CASTIGOS_TOTAL','PERSONA_JURIDICA']:
    df_deb[col] = 0
df_deb['MTO_COMPRAS_NAL']  = df_deb['MTO_COMPRAS_CREDITO']
df_deb['VIGENTES_MES']     = df_deb['VIGENTES_FECHA_CORTE']
df_deb = df_deb[df_deb['FRANQUICIA'].isin(['VISA','MASTERCARD'])].copy()
# Normalizar MES a string YYYY-MM — garantiza formato consistente con df_dash
df_deb['MES'] = df_deb['MES'].astype(str).str[:7]
df_deb = df_deb[['MES','ENTIDAD','FRANQUICIA','MTO_COMPRAS_CREDITO','MTO_COMPRAS_NAL',
                  'MTO_COMPRAS_EXT','NUM_COMPRAS_NAL','NUM_COMPRAS_EXT',
                  'VIGENTES_FECHA_CORTE','CANCELADAS','VIGENTES_MES',
                  'SALDO_CARTERA','CUPO_NO_UTILIZADO','INTERESES_CORRIENTES',
                  'INTERESES_MORA','CASTIGOS_TOTAL','PERSONA_JURIDICA']].copy()
df_deb = df_deb.where(pd.notna(df_deb), None)
debito_json = df_deb.to_json(orient='records', force_ascii=False)

# ── Dataset TOTAL: crédito completo (V+MC+Amex+Diners) + débito (V+MC) ─────────
# Include ALL franquicias from credito so Amex/Diners appear in Total denominator
df_cred_tot = df_dash.copy()  # all franquicias: VISA, MASTERCARD, AMEX, DINERS
for col in df_deb.columns:
    if col not in df_cred_tot.columns: df_cred_tot[col] = 0
for col in df_cred_tot.columns:
    if col not in df_deb.columns: df_deb[col] = 0
df_total = pd.concat([df_cred_tot, df_deb[df_cred_tot.columns]], ignore_index=True)
df_total['MES'] = df_total['MES'].astype(str).str[:7]
df_total = df_total.where(pd.notna(df_total), None)
total_json = df_total.to_json(orient='records', force_ascii=False)

# ── Diagnóstico ───────────────────────────────────────────────────────────────
ultimo_d = df_deb['MES'].max()
ult_grp = df_deb[df_deb['MES']==ultimo_d].groupby('FRANQUICIA')['MTO_COMPRAS_CREDITO'].sum()
tot_d = ult_grp.sum()
print(f"✅ Débito {ultimo_d}: ${tot_d/1e12:.2f}T COP (monto real por emisor)")
print(f"   MES dtype: {df_deb['MES'].dtype}, sample: {df_deb['MES'].iloc[0]}")
print(f"   Nequi rows: {(df_deb['ENTIDAD']=='NEQUI (Bancolombia)').sum()}")
print(f"   Entidades únicas: {df_deb['ENTIDAD'].nunique()}")
for f,v in ult_grp.items(): print(f"   {f}: ${v/1e12:.2f}T ({v/tot_d*100:.1f}% MS)")
print(f"   Nequi separado: {'NEQUI (Bancolombia)' in df_deb['ENTIDAD'].values}")
ult_tot = df_total[df_total['MES']==ultimo_d].groupby('FRANQUICIA')['MTO_COMPRAS_CREDITO'].sum()
tot_t = ult_tot.sum()
print(f"\n✅ Total {ultimo_d}: ${tot_t/1e12:.2f}T COP (crédito + débito)")
for f,v in ult_tot.items(): print(f"   {f}: ${v/1e12:.2f}T ({v/tot_t*100:.1f}% MS)")

# ── 6.1c Preparar datos de retiros débito (ratio retiros/compras por banco) ────
EXCLUIR_REDES_RET = [
    'REDEBAN S.A.',
    'MASTERCARD COLOMBIA ADMINISTRADORA S.A.',
    'CREDIBANCO S.A. PUDIENDO SIN PERDER SU NATURALEZA UTILIZAR LA SIGLA CREDIBANCO',
    'VISA COLOMBIA SUPPORT SERVICES SOCIEDAD ANONIMA',
    'VISIONAMOS SISTEMA DE PAGO COOPERATIVO',
]

def _get_deb_serie(descripcion):
    df = df_raw[
        (df_raw['DESCRIPCION'] == descripcion) &
        (~df_raw['NOMBREENTIDAD'].isin(EXCLUIR_REDES_RET))
    ].copy()
    df['MES'] = pd.to_datetime(df['FECHACORTE'], format='%d/%m/%Y', dayfirst=True, errors='coerce').dt.strftime('%Y-%m')
    df['val'] = pd.to_numeric(df['TOTAL_TARJETAS'].astype(str).str.replace(',','', regex=False), errors='coerce').fillna(0)
    return df.groupby(['MES','NOMBREENTIDAD'])['val'].sum().reset_index()

df_ret_raw  = _get_deb_serie('Monto de transacciones por retiros con tarjetas débito')
df_comp_raw = _get_deb_serie('Monto de transacciones por compras con tarjetas débito')

# Merge retiros y compras por banco/mes
df_ratio = df_ret_raw.merge(df_comp_raw, on=['MES','NOMBREENTIDAD'], suffixes=('_ret','_comp'))
df_ratio['MTO_RETIROS']  = df_ratio['val_ret']
df_ratio['MTO_COMPRAS']  = df_ratio['val_comp']
df_ratio['MTO_TOTAL_DEB'] = df_ratio['val_ret'] + df_ratio['val_comp']
df_ratio['RATIO_RETIROS'] = df_ratio['MTO_RETIROS'] / df_ratio['MTO_TOTAL_DEB'].replace(0, 1) * 100

# Asignar franquicia via TII (mismo approach que df_deb)
# Merge con df_tii_grp para obtener PROP por banco/mes
df_tii_prop = df_tii_grp[['MES','ENTIDAD','FRANQUICIA','PROP']].copy()
df_ratio = df_ratio.rename(columns={'NOMBREENTIDAD':'ENTIDAD'})
df_ret_franq = df_ratio.merge(df_tii_prop, on=['MES','ENTIDAD'], how='left')
# Para bancos sin TII, asumir 100% MASTERCARD (mayoría de emisores)
df_ret_franq['FRANQUICIA'] = df_ret_franq['FRANQUICIA'].fillna('MASTERCARD')
df_ret_franq['PROP'] = df_ret_franq['PROP'].fillna(1.0)
# Nequi: separar igual que en df_deb
banc_mask_r = df_ret_franq['ENTIDAD'].str.contains('BANCOLOMBIA', na=False)
nequi_r = df_ret_franq[banc_mask_r & (df_ret_franq['FRANQUICIA']=='VISA')].copy()
nequi_r['ENTIDAD'] = 'NEQUI (Bancolombia)'
banc_mc_r = df_ret_franq[banc_mask_r & (df_ret_franq['FRANQUICIA']=='MASTERCARD')].copy()
otros_r = df_ret_franq[~banc_mask_r].copy()
df_ret_franq = pd.concat([nequi_r, banc_mc_r, otros_r], ignore_index=True)
df_ret_franq['MES'] = df_ret_franq['MES'].astype(str).str[:7]
df_ret_franq = df_ret_franq[df_ret_franq['FRANQUICIA'].isin(['VISA','MASTERCARD'])].copy()
df_ret_franq = df_ret_franq.where(pd.notna(df_ret_franq), None)

retiros_json = df_ret_franq[['MES','ENTIDAD','FRANQUICIA',
    'MTO_RETIROS','MTO_COMPRAS','MTO_TOTAL_DEB','RATIO_RETIROS']].to_json(orient='records', force_ascii=False)

ultimo_r = df_ret_franq['MES'].max()
mkt_ret = df_ret_franq[df_ret_franq['MES']==ultimo_r]['MTO_RETIROS'].sum()
mkt_comp = df_ret_franq[df_ret_franq['MES']==ultimo_r]['MTO_COMPRAS'].sum()
mkt_ratio = mkt_ret/(mkt_ret+mkt_comp)*100
print(f"✅ Retiros débito {ultimo_r}: ratio mercado = {mkt_ratio:.1f}% ({len(df_ret_franq['ENTIDAD'].unique())} bancos)")


In [ ]:
# ── 6.2 Generar dashboard HTML con datos incrustados ──────────────────────────
# Estrategia: sustituir 4 placeholders planos con str.replace()
# NO usar f-strings sobre el template para evitar conflictos con llaves de JS

TEMPLATE = """<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SFC · Market Intelligence Colombia</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --visa:#1A1F71; --visa-light:#E8EAF6;
  --mc:#CC2200;   --mc-light:#FDE8E4;
  --amex:#8E9296; --amex-light:#F0F1F2;
  --diners:#334155;
  --bg:#F7F8FA; --card:#FFFFFF; --border:#E4E7EC;
  --text:#101828; --muted:#667085; --faint:#98A2B3;
  --green:#027A48; --green-bg:#ECFDF3;
  --red:#B42318;   --red-bg:#FEF3F2;
  --visa-font:"Visa Dialect","Helvetica Neue",Arial,sans-serif;
}
body{font-family:var(--visa-font);background:var(--bg);color:var(--text);font-size:14px;line-height:1.5}
.page{max-width:1140px;margin:0 auto;padding:28px 20px}
/* Header */
.hdr{display:none}
.hdr-title{font-size:19px;font-weight:700;letter-spacing:-0.3px}
.hdr-sub{font-size:11px;color:var(--muted);margin-top:3px}
.banner{background:#0A1172;margin:-28px -20px 20px;padding:18px 32px 16px;border-bottom:3px solid #F7A600}
.banner-eyebrow{font-size:9px;font-weight:700;letter-spacing:.16em;text-transform:uppercase;color:rgba(255,255,255,0.45);margin-bottom:4px}
.banner-title{font-size:20px;font-weight:800;color:#ffffff;letter-spacing:-.2px;line-height:1.2}
.banner-title .accent{color:#F7A600;font-weight:400}
.banner-sub{display:none}
.banner-right{display:none}
.banner-date{display:none}
/* Filters bar */
.filters{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin-bottom:20px;padding:12px 14px;background:var(--card);border:1px solid var(--border);border-radius:10px}
.msel{position:relative;display:inline-block}
.msel-btn{font-family:inherit;font-size:12px;padding:5px 28px 5px 10px;border:1px solid var(--border);border-radius:6px;background:var(--card);color:var(--text);cursor:pointer;min-width:140px;text-align:left;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;appearance:none;-webkit-appearance:none}
.msel-btn::after{content:"▾";position:absolute;right:8px;top:50%;transform:translateY(-50%);pointer-events:none;font-size:10px;color:var(--muted)}
.msel-btn:hover,.msel-btn:focus{border-color:var(--visa);outline:none}
.msel-btn.has-sel{border-color:var(--visa);color:var(--visa);font-weight:600}
.msel-drop{display:none;position:absolute;top:calc(100% + 4px);left:0;min-width:200px;max-width:300px;max-height:260px;overflow-y:auto;background:var(--card);border:1px solid var(--border);border-radius:8px;box-shadow:0 4px 16px rgba(0,0,0,0.12);z-index:999;padding:4px 0}
.msel.open .msel-drop{display:block}
.msel-item{display:flex;align-items:center;gap:8px;padding:6px 12px;font-size:12px;cursor:pointer;color:var(--text);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.msel-item:hover{background:var(--bg)}
.msel-item input{width:14px;height:14px;accent-color:var(--visa);cursor:pointer;flex-shrink:0}
.msel-item.all-item{font-weight:600;border-bottom:1px solid var(--border);margin-bottom:4px;padding-bottom:8px}
.filter-label{font-size:11px;font-weight:600;color:var(--muted);text-transform:uppercase;letter-spacing:.05em;white-space:nowrap}
select,button.tab{font-family:inherit;font-size:12px;padding:5px 10px;border:1px solid var(--border);border-radius:6px;background:var(--card);color:var(--text);cursor:pointer;outline:none}
select:hover,button.tab:hover{border-color:var(--visa)}
button.tab.active{background:var(--visa);color:#fff;border-color:var(--visa)}
.banner .tab.active{background:#ffffff;color:var(--visa);border-color:#ffffff}
.banner .tab{color:rgba(255,255,255,0.7);border-color:rgba(255,255,255,0.3)}
.banner .tab:hover{background:rgba(255,255,255,0.15)}
.sep{width:1px;height:20px;background:var(--border);margin:0 4px}
/* Section labels */
.slabel{font-size:10px;font-weight:700;letter-spacing:.09em;text-transform:uppercase;color:var(--faint);margin:24px 0 10px}
/* KPI grid */
.kpi-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:10px}
.kpi{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:14px 16px;border-left:3px solid}
.kpi-lbl{font-size:11px;color:var(--muted);margin-bottom:5px}
.kpi-val{font-size:23px;font-weight:700}
.kpi-delta{font-size:11px;margin-top:4px;display:flex;align-items:center;gap:5px}
.chip{display:inline-flex;align-items:center;padding:2px 7px;border-radius:4px;font-size:11px;font-weight:600}
.chip-up{background:var(--green-bg);color:var(--green)}
.chip-down{background:var(--red-bg);color:var(--red)}
.chip-neu{background:#F2F4F7;color:var(--muted)}
/* Card */
.card{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:16px 18px}
.card-title{font-size:13px;font-weight:700;margin-bottom:2px}
.card-sub{font-size:11px;color:var(--muted);margin-bottom:10px}
/* Layouts */
.row-2{display:grid;grid-template-columns:2fr 1fr;gap:14px;margin-top:14px}
.row-3{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-top:14px}
.row-2s{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-top:14px}
.row-4{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-top:14px}
/* Chart */
.chart-wrap{position:relative;width:100%}
/* Legend */
.leg-row{display:flex;flex-wrap:wrap;gap:12px;margin-bottom:8px}
.leg{display:flex;align-items:center;gap:5px;font-size:11px;color:var(--muted)}
.leg-sq{width:9px;height:9px;border-radius:2px;flex-shrink:0}
/* Snapshot */
.snap-grid{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.snap-card{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:18px 20px}
.snap-ms{font-size:34px;font-weight:800;margin:5px 0 2px}
.snap-sub{font-size:11px;color:var(--muted)}
.snap-div{border:none;border-top:1px solid var(--border);margin:12px 0}
.snap-tbl{width:100%;border-collapse:collapse}
.snap-tbl td{padding:5px 0;font-size:12px;vertical-align:middle}
.snap-tbl td:last-child{text-align:right}
.td-lbl{color:var(--muted)}
/* Bar segments */
.bar-bg{background:#F2F4F7;border-radius:3px;height:7px;overflow:hidden;margin-top:3px}
.bar-fill{height:7px;border-radius:3px}
.intl-bar{display:flex;border-radius:4px;overflow:hidden;height:14px}
@media(max-width:720px){
  .kpi-grid,.row-2,.row-3,.row-2s,.row-4,.snap-grid{grid-template-columns:1fr}
}
</style>
</head>
<body>
<div class="page">

<!-- HEADER -->
<div class="hdr"></div>
<div class="banner">
  <div class="banner-eyebrow">Market Intelligence Report &nbsp;&middot;&nbsp; __ULTIMO_MES__</div>
  <div class="banner-title">Visa Colombia &nbsp;<span class="accent">&mdash;</span>&nbsp; Market Intelligence</div>
  <div style="font-size:11px;color:#A0AAC0;margin-top:4px;letter-spacing:.03em">Elaborado por Daniel Casta&ntilde;eda &nbsp;&middot;&nbsp; Business Development</div>
</div>

<!-- FILTERS -->
<div class="filters">

  <!-- Banco -->
  <span class="filter-label">Banco</span>
  <div class="msel" id="msel-banco">
    <button class="msel-btn" onclick="toggleMsel('banco')" id="msel-banco-btn">Todos los bancos</button>
    <div class="msel-drop" id="msel-banco-drop">
      <label class="msel-item all-item"><input type="checkbox" id="msel-banco-all" checked onchange="mselAll('banco')"><span>Todos</span></label>
      <div id="msel-banco-items"></div>
    </div>
  </div>

  <span class="sep"></span>

  <!-- Año -->
  <span class="filter-label">Año</span>
  <div class="msel" id="msel-year">
    <button class="msel-btn" onclick="toggleMsel('year')" id="msel-year-btn">Todos los años</button>
    <div class="msel-drop" id="msel-year-drop">
      <label class="msel-item all-item"><input type="checkbox" id="msel-year-all" checked onchange="mselAll('year')"><span>Todos</span></label>
      <div id="msel-year-items"></div>
    </div>
  </div>

  <span class="sep"></span>

  <!-- Mes -->
  <span class="filter-label">Mes</span>
  <div class="msel" id="msel-mes">
    <button class="msel-btn" onclick="toggleMsel('mes')" id="msel-mes-btn">Todos los meses</button>
    <div class="msel-drop" id="msel-mes-drop">
      <label class="msel-item all-item"><input type="checkbox" id="msel-mes-all" checked onchange="mselAll('mes')"><span>Todos</span></label>
        <label class="msel-item"><input type="checkbox" value="01" onchange="mselChange('mes')"><span>Enero</span></label>
        <label class="msel-item"><input type="checkbox" value="02" onchange="mselChange('mes')"><span>Febrero</span></label>
        <label class="msel-item"><input type="checkbox" value="03" onchange="mselChange('mes')"><span>Marzo</span></label>
        <label class="msel-item"><input type="checkbox" value="04" onchange="mselChange('mes')"><span>Abril</span></label>
        <label class="msel-item"><input type="checkbox" value="05" onchange="mselChange('mes')"><span>Mayo</span></label>
        <label class="msel-item"><input type="checkbox" value="06" onchange="mselChange('mes')"><span>Junio</span></label>
        <label class="msel-item"><input type="checkbox" value="07" onchange="mselChange('mes')"><span>Julio</span></label>
        <label class="msel-item"><input type="checkbox" value="08" onchange="mselChange('mes')"><span>Agosto</span></label>
        <label class="msel-item"><input type="checkbox" value="09" onchange="mselChange('mes')"><span>Septiembre</span></label>
        <label class="msel-item"><input type="checkbox" value="10" onchange="mselChange('mes')"><span>Octubre</span></label>
        <label class="msel-item"><input type="checkbox" value="11" onchange="mselChange('mes')"><span>Noviembre</span></label>
        <label class="msel-item"><input type="checkbox" value="12" onchange="mselChange('mes')"><span>Diciembre</span></label>
    </div>
  </div>

  <span class="sep"></span>
  <span class="filter-label">Período</span>
  <button class="tab active" id="period-btn-mes" onclick="setView('mensual',this)">Mensual</button>
  <button class="tab" id="period-btn-ano" onclick="setView('anual',this)">Anual</button>
  <span class="sep"></span>
  <span class="filter-label">Franquicia</span>
  <div class="msel" id="msel-franq">
    <button class="msel-btn" onclick="toggleMsel('franq')" id="msel-franq-btn">Todas</button>
    <div class="msel-drop" id="msel-franq-drop">
      <label class="msel-item all-item"><input type="checkbox" id="msel-franq-all" checked onchange="mselAll('franq')"><span>Todas</span></label>
      <div id="msel-franq-items"></div>
    </div>
  </div>
  <span class="sep"></span>
  <span class="filter-label">Tipo</span>
  <button class="tab active" id="tipo-btn-cred" onclick="setTipo('credito',this)">Crédito</button>
  <button class="tab" id="tipo-btn-deb" onclick="setTipo('debito',this)">Débito</button>
  <button class="tab" id="tipo-btn-tot" onclick="setTipo('total',this)">Total</button>
  <span style="flex:1"></span>
  <span id="filter-note" style="font-size:11px;color:var(--faint)"></span>
</div>
<!-- VIZ 1 — MS 100% Horizontal Bar -->
<div class="slabel">snapshot comparativo — último período</div>
<div class="snap-grid" id="snap-grid"></div>

<div class="slabel">market share — facturación por franquicia</div>
<div class="card">
  <div class="card-title" id="ms-title">Evolución mensual del market share</div>
  <div class="card-sub">Participación porcentual sobre total de compras. Cada barra suma 100%.</div>
  <div class="leg-row">
    <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
    <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
    <span class="leg"><span class="leg-sq" style="background:var(--amex)"></span>Amex</span>
    <span class="leg"><span class="leg-sq" style="background:var(--diners)"></span>Diners</span>
  </div>
  <div class="chart-wrap" style="height:240px"><canvas id="msChart"></canvas></div>
</div>

<!-- VIZ 2 + 3 -->
<div class="card" style="margin-top:0">
  <div class="card-title">Facturación</div>
  <div class="card-sub">Monto compras nacionales, exterior y total por franquicia (COP)</div>
  <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;margin-bottom:8px">
    <div class="leg-row" style="margin-bottom:0">
      <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
      <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
    </div>
    <div style="display:flex;gap:4px;flex-wrap:wrap">
      <button class="tab active" id="vol-tab-tot" onclick="setVolTab('tot',this)">Total</button>
      <button class="tab" id="vol-tab-nal" onclick="setVolTab('nal',this)">Nacional</button>
      <button class="tab" id="vol-tab-ext" onclick="setVolTab('ext',this)">Exterior</button>
      <span style="width:1px;background:var(--border);margin:0 2px"></span>
      <button class="tab active" id="vol-period-mes" onclick="setVolPeriod('mensual',this)">Mensual</button>
      <button class="tab" id="vol-period-ano" onclick="setVolPeriod('anual',this)">Anual</button>
    </div>
  </div>
  <div class="chart-wrap" style="height:190px"><canvas id="volChart"></canvas></div>
</div>

<!-- VIZ transacciones + tarjetas vigentes -->
<div class="card" style="margin-top:14px">
  <div class="card-title">Cantidad de transacciones</div>
  <div class="card-sub" id="txn-sub">Compras nacionales + exterior</div>
  <div class="leg-row">
    <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
    <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
    <span class="leg" id="txn-yoy-visa" style="margin-left:auto"></span>
    <span class="leg" id="txn-yoy-mc"></span>
  </div>
  <div class="chart-wrap" style="height:220px"><canvas id="txnChart"></canvas></div>
</div>
<div class="card" style="margin-top:14px">
  <div class="card-title">Tarjetas vigentes</div>
  <div class="card-sub" id="vig-sub">Total tarjetas vigentes al corte del período</div>
  <div class="leg-row">
    <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
    <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
    <span class="leg" id="vig-yoy-visa" style="margin-left:auto"></span>
    <span class="leg" id="vig-yoy-mc"></span>
  </div>
  <div class="chart-wrap" style="height:220px"><canvas id="vigChart"></canvas></div>
</div>
<div class="card" style="margin-top:14px">
  <div class="card-title">Ticket promedio</div>
  <div class="card-sub">Miles de pesos COP por compra nacional</div>
  <div class="leg-row">
    <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
    <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
  </div>
  <div class="chart-wrap" style="height:220px"><canvas id="ticketChart"></canvas></div>
</div>
<div class="card" style="margin-top:14px">
  <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;margin-bottom:8px">
    <div>
      <div class="card-title" id="yoy-title">Crecimiento YoY — Facturación</div>
      <div class="card-sub" id="yoy-sub">Variación anual · Visa vs Mastercard · últimos 18 meses · basado en crédito</div>
    </div>
    <div style="display:flex;gap:4px;flex-wrap:wrap">
      <button class="tab active" id="yoy-tab-fact" onclick="setYoyTab('fact',this)">Facturación</button>
      <button class="tab" id="yoy-tab-txn"  onclick="setYoyTab('txn',this)">Transacciones</button>
      <button class="tab" id="yoy-tab-vig"  onclick="setYoyTab('vig',this)">Tarjetas</button>
      <button class="tab" id="yoy-tab-tkt"  onclick="setYoyTab('tkt',this)">Ticket</button>
    </div>
  </div>
  <div class="leg-row">
    <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
    <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
  </div>
  <div class="chart-wrap" style="height:220px"><canvas id="yoyChart"></canvas></div>
</div>


<!-- VIZ desglose periodo -->
<div class="slabel">desglose del último período</div>
<div class="row-2s">
  <div class="card">
    <div class="card-title">Contribución al cambio en MS</div>
    <div class="card-sub" id="contrib-sub">Δ pp vs período anterior</div>
    <div id="contrib-body"></div>
  </div>
  <div class="card">
    <div class="card-title">Nacional vs exterior</div>
    <div class="card-sub">Mix nacional vs exterior — último mes</div>
    <div id="intl-body" style="margin-top:4px"></div>
  </div>
</div>

<!-- VIZ 10 — Participación por banco (horizontal stacked) -->
<div class="slabel">participación por banco — último mes disponible</div>
<div class="card">
  <div class="card-title">Monto de compras por banco y franquicia</div>
  <div class="card-sub">Barras horizontales apiladas ordenadas por volumen total.</div>
  <div class="leg-row">
    <span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span>
    <span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span>
    <span class="leg"><span class="leg-sq" style="background:var(--amex)"></span>Amex</span>
    <span class="leg"><span class="leg-sq" style="background:var(--diners)"></span>Diners</span>
  </div>
  <div class="chart-wrap" id="banco-wrap"><canvas id="bancoChart"></canvas></div>
</div>

<!-- VIZ contribución por banco -->
<div class="slabel">contribución al cambio en market share — por banco</div>
<div class="row-2s">
  <div class="card">
    <div class="card-title">Top bancos por ganancia/pérdida de MS — Visa</div>
    <div class="card-sub" id="contrib-banco-visa-sub">Δ pp vs período anterior</div>
    <div id="contrib-banco-visa"></div>
  </div>
  <div class="card">
    <div class="card-title">Top bancos por ganancia/pérdida de MS — Mastercard</div>
    <div class="card-sub" id="contrib-banco-mc-sub">Δ pp vs período anterior</div>
    <div id="contrib-banco-mc"></div>
  </div>
</div>

<!-- VIZ salud de cartera -->
<div class="slabel">salud de cartera · últimos 24 meses</div>
<div class="card" style="margin-top:0">
  <div class="card-title">Utilización de cupo</div>
  <div class="card-sub">Saldo / (Saldo + Cupo disponible)</div>
  <div class="leg-row"><span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span><span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span></div>
  <div class="chart-wrap" style="height:240px"><canvas id="utilizChart"></canvas></div>
</div>
<div class="card" style="margin-top:14px">
  <div class="card-title">Ratio de mora</div>
  <div class="card-sub">Intereses mora / Intereses corrientes</div>
  <div class="leg-row"><span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span><span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span></div>
  <div class="chart-wrap" style="height:240px"><canvas id="moraChart"></canvas></div>
</div>
<div class="card" style="margin-top:14px">
  <div class="card-title">Churn rate implícito</div>
  <div class="card-sub">Tarjetas canceladas / Vigentes en el mes</div>
  <div class="leg-row"><span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span><span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span></div>
  <div class="chart-wrap" style="height:240px"><canvas id="churnChart"></canvas></div>
</div>
<div class="card" style="margin-top:14px">
  <div class="card-title">Tasa de castigo</div>
  <div class="card-sub">Castigos totales / Saldo de cartera</div>
  <div class="leg-row"><span class="leg"><span class="leg-sq" style="background:var(--visa)"></span>Visa</span><span class="leg"><span class="leg-sq" style="background:var(--mc)"></span>Mastercard</span></div>
  <div class="chart-wrap" style="height:240px"><canvas id="castigoChart"></canvas></div>
</div>

<!-- VIZ 11 — Snapshot comparativo -->



<!-- VIZ RETIROS — Ratio retiros / (retiros + compras) débito -->
<div class="card">
  <div class="card-title">Uso de Cajeros ATM — Débito</div>
  <div class="card-sub" id="retiros-sub">Retiros / (Retiros + Compras) por banco · últimos 24 meses</div>

  <div id="retiros-wrap" style="height:520px;position:relative">
    <canvas id="retirosChart"></canvas>
  </div>
</div>

<!-- VIZ 12 — Mapa de Calor Banco x Mes -->
<div class="slabel" style="margin-top:28px">mapa de calor — variación yoy de market share por banco</div>
<div class="card">
  <div class="card-title">Mapa de Calor: Market Share YoY por Banco</div>
  <div class="card-sub">Variación anual del market share por banco · verde = ganó pp · rojo = perdió pp</div>
  <div id="heatmap-franq-tabs" style="display:flex;gap:4px;margin-bottom:10px">
    <button class="tab active" onclick="setHeatmapFranq('VISA',this)">Visa</button>
    <button class="tab" onclick="setHeatmapFranq('MASTERCARD',this)">Mastercard</button>
  </div>
  <div id="heatmap-container" style="overflow-x:auto"></div>
</div>

<!-- VIZ 13 — Cuadrante Competitivo -->
<div class="slabel" style="margin-top:28px">análisis de cohorte — posicionamiento competitivo por banco</div>
<div class="card">
  <div class="card-title">Cuadrante Competitivo: Market Share vs Crecimiento YoY</div>
  <div class="card-sub">Eje X = market share actual · Eje Y = crecimiento YoY facturación · tamaño = volumen total</div>
  <div id="cohort-franq-tabs" style="display:flex;gap:4px;margin-bottom:10px">
    <button class="tab active" onclick="setCohortFranq('VISA',this)">Visa</button>
    <button class="tab" onclick="setCohortFranq('MASTERCARD',this)">Mastercard</button>
  </div>
  <div class="chart-wrap" style="height:420px"><canvas id="cohortChart"></canvas></div>
</div>


<div style="margin-top:28px;font-size:11px;color:var(--faint);text-align:center">
  Superintendencia Financiera de Colombia · datos.gov.co · __ULTIMO_MES__
</div>

</div><!-- /page -->

<script>
const RAW_CREDITO = __DATA_JSON__;
const RAW_DEBITO  = __DEBITO_JSON__;
const RAW_TOTAL   = __TOTAL_JSON__;
// ALL_BANCOS: union of all entities across all tipos, sorted by fmtBank name
const ALL_BANCOS = __BANCOS_JSON__;
const RAW_RETIROS = __RETIROS_JSON__;
let currentTipo = "credito";
let RAW = RAW_CREDITO;
const BANCOS = __BANCOS_JSON__;

const VISA  = "#1A1F71";
const MC    = "#CC2200";
const AMEX  = "#8E9296";
const DINERS= "#334155";
const C = {VISA, MASTERCARD: MC, "AMERICAN EXPRESS": AMEX, DINERS, "OTRAS TARJETAS DE CREDITO": "#94A3B8"};
// ALL_F computed dynamically from current RAW so débito/total only sees VISA+MC
function getActiveFranqs() {
  return [...new Set(RAW.map(r => r.FRANQUICIA))].filter(f =>
    ["VISA","MASTERCARD","AMERICAN EXPRESS","DINERS"].includes(f)
  );
}
const ALL_F_CRED = ["VISA","MASTERCARD","AMERICAN EXPRESS","DINERS","OTRAS TARJETAS DE CREDITO"];
let ALL_F = ALL_F_CRED;

let currentView = "mensual";
let heatmapFranq = "VISA";
let cohortFranq  = "VISA";
let volTab = "tot";
let volPeriod = "mensual";
let yoyTab = "fact";
let CHARTS = {};

// ── Populate bank dropdown ────────────────────────────────────────────────────
// Banco dropdown — dynamically populated based on current RAW
function populateBancos() {
  const bancoDrop = document.getElementById("msel-banco-items");
  if (!bancoDrop) return;
  bancoDrop.innerHTML = "";
  // Reset "Todos" checkbox
  const allCb = document.getElementById("msel-banco-all");
  if (allCb) allCb.checked = true;
  mselUpdateBtn("banco");
  // Show only entities present in current RAW (adjusts to tipo)
  const activeEntities = [...new Set(RAW.map(r => r.ENTIDAD))].sort(
    (a,b) => fmtBank(a).localeCompare(fmtBank(b))
  );
  activeEntities.forEach(b => {
    const lbl = document.createElement("label");
    lbl.className = "msel-item";
    const disp = fmtBank(b);
    const safeVal = b.replace(/"/g, '&quot;');
    const safeTitle = b.replace(/"/g, '&quot;');
    lbl.innerHTML = `<input type="checkbox" value="${safeVal}" onchange="mselChange('banco')"><span title="${safeTitle}">${disp}</span>`;
    bancoDrop.appendChild(lbl);
  });
}
populateBancos();
// Populate franquicia dropdown — uses ALL_F_CRED (all franquicias)
function populateFranqs() {
  const drop = document.getElementById("msel-franq-items");
  if (!drop) return;
  drop.innerHTML = "";
  const allCb = document.getElementById("msel-franq-all");
  if (allCb) allCb.checked = true;
  mselUpdateBtn("franq");
  // Show franquicias present in current RAW
  const activeFranqs = [...new Set(RAW.map(r => r.FRANQUICIA))]
    .filter(f => ALL_F_CRED.includes(f))
    .sort();
  const labels = {"VISA":"Visa","MASTERCARD":"Mastercard","AMERICAN EXPRESS":"Amex",
                  "DINERS":"Diners","OTRAS TARJETAS DE CREDITO":"Otras"};
  activeFranqs.forEach(f => {
    const lbl = document.createElement("label");
    lbl.className = "msel-item";
    lbl.innerHTML = `<input type="checkbox" value="${f}" onchange="mselChange('franq')"><span>${labels[f]||f}</span>`;
    drop.appendChild(lbl);
  });
}
populateFranqs();

// Populate year dropdown — uses RAW_CREDITO (years same across tipos)
function populateYears() {
  const yearDrop = document.getElementById("msel-year-items");
  if (!yearDrop) return;
  yearDrop.innerHTML = "";
  const allYears = [...new Set(RAW_CREDITO.map(r => r.MES.slice(0,4)))].sort().reverse();
  allYears.forEach(y => {
    const lbl = document.createElement("label");
    lbl.className = "msel-item";
    lbl.innerHTML = `<input type="checkbox" value="${y}" onchange="mselChange('year')"><span>${y}</span>`;
    yearDrop.appendChild(lbl);
  });
}
populateYears();
// Close dropdowns when clicking outside
document.addEventListener("click", e => {
  if (!e.target.closest(".msel")) document.querySelectorAll(".msel.open").forEach(el => el.classList.remove("open"));
});

// ── Helpers ───────────────────────────────────────────────────────────────────
function fmtB(n) {
  if (n == null || isNaN(n) || n === 0) return "—";
  const abs = Math.abs(n);
  if (abs >= 1e12) return "$" + (n/1e12).toFixed(2) + " T";
  if (abs >= 1e9)  return "$" + (n/1e9).toFixed(1)  + " B";
  if (abs >= 1e6)  return "$" + (n/1e6).toFixed(0)  + " M";
  return "$" + n.toFixed(0);
}
// Scale helper for chart Y axis
function chartScale(vals) {
  const mx = Math.max(...vals.filter(v=>v!=null&&!isNaN(v)), 1);
  if (mx >= 1e12) return {div:1e12, sfx:"T"};
  if (mx >= 1e9)  return {div:1e9,  sfx:"B"};
  if (mx >= 1e6)  return {div:1e6,  sfx:"M"};
  return {div:1, sfx:""};
}
function viewPeriods(periods) {
  const years = getPillVals("year");
  const meses = getPillVals("mes");
  let vp = periods;
  if (years) vp = vp.filter(p => years.includes(p.slice(0,4)));
  if (meses) vp = vp.filter(p => meses.includes(p.slice(5,7)));
  if (!years && !meses) {
    if (currentView === "anual") return periods;
    return periods.slice(-24);
  }
  return vp;
}
function viewLabels(periods) {
  const vp = viewPeriods(periods);
  // Always show full YYYY-MM (or YYYY for annual) so axis is unambiguous
  if (currentView === "anual") return vp;
  return vp.map(p => p.length >= 7 ? p.slice(0,7) : p);
}
function pct(n, d=1) { return (n == null || isNaN(n)) ? "—" : (n*100).toFixed(d) + "%"; }
function chip(v, sfx, invert=false) {
  if (v == null || isNaN(v)) return "";
  const pos = invert ? v < 0 : v > 0;
  const cls = Math.abs(v) < 0.05 ? "chip-neu" : pos ? "chip-up" : "chip-down";
  return `<span class="chip ${cls}">${v > 0 ? "+" : ""}${v.toFixed(1)}${sfx}</span>`;
}
function mkChart(id, cfg) {
  if (CHARTS[id]) { CHARTS[id].destroy(); }
  const el = document.getElementById(id);
  if (!el) return null;
  CHARTS[id] = new Chart(el, cfg);
  return CHARTS[id];
}
function lineOpts(fmt, extra={}) {
  return {
    responsive: true, maintainAspectRatio: false,
    layout: { padding: { top:10, right:10, bottom:2, left:2 } },
    interaction: { mode:"index", intersect:false },
    plugins: {
      legend: {display:false},
      tooltip: {
        mode:"index", intersect:false,
        callbacks: { label: c => `${c.dataset.label}: ${c.parsed.y != null ? c.parsed.y.toFixed(2)+fmt : "—"}` }
      }
    },
    scales: {
      x:{ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}},grid:{display:false}},
      y: { grace:"8%", ticks:{callback: v => v.toFixed(1)+fmt, font:{size:9}}, grid:{color:"#F2F4F7"} }
    },
    ...extra
  };
}

// ── Filter raw data ───────────────────────────────────────────────────────────
function toggleMsel(group) {
  const el = document.getElementById("msel-"+group);
  const wasOpen = el.classList.contains("open");
  document.querySelectorAll(".msel.open").forEach(x => x.classList.remove("open"));
  if (!wasOpen) el.classList.add("open");
}

function mselAll(group) {
  const allCb = document.getElementById("msel-"+group+"-all");
  const drop = document.getElementById("msel-"+group+"-drop");
  drop.querySelectorAll("input[type=checkbox]:not(#msel-"+group+"-all)").forEach(cb => cb.checked = false);
  allCb.checked = true;
  mselUpdateBtn(group);
  renderAll();
}

function mselChange(group) {
  const allCb = document.getElementById("msel-"+group+"-all");
  const drop = document.getElementById("msel-"+group+"-drop");
  const checked = [...drop.querySelectorAll("input[type=checkbox]:not(#msel-"+group+"-all):checked")];
  allCb.checked = checked.length === 0;
  mselUpdateBtn(group);
  renderAll();
}

function mselGetVals(group) {
  const allCb = document.getElementById("msel-"+group+"-all");
  if (allCb?.checked) return null;
  const drop = document.getElementById("msel-"+group+"-drop");
  if (!drop) return null;
  const vals = [...drop.querySelectorAll("input[type=checkbox]:not(#msel-"+group+"-all):checked")]
    .map(cb => cb.value.replace(/&quot;/g, '"'));
  return vals.length > 0 ? vals : null;
}

function mselUpdateBtn(group) {
  const btn = document.getElementById("msel-"+group+"-btn");
  const vals = mselGetVals(group);
  const labels = {"banco":"Todos los bancos","year":"Todos los años","mes":"Todos los meses","franq":"Todas"};
  if (!vals) { btn.textContent = labels[group]; btn.classList.remove("has-sel"); }
  else { btn.textContent = vals.length === 1 ? fmtBank(vals[0]).slice(0,22) : vals.length+" seleccionados"; btn.classList.add("has-sel"); }
}

// Alias for backward compat with getPillVals references
function getPillVals(group) { return mselGetVals(group); }

function filtered() {
  let rows = RAW;
  const bancos = getPillVals("banco");
  if (bancos) rows = rows.filter(r => bancos.includes(r.ENTIDAD));
  const franqs = getPillVals("franq");
  if (franqs) rows = rows.filter(r => franqs.includes(r.FRANQUICIA));
  const years = getPillVals("year");
  if (years) rows = rows.filter(r => years.includes(r.MES.slice(0,4)));
  const meses = getPillVals("mes");
  if (meses) rows = rows.filter(r => meses.includes(r.MES.slice(5,7)));
  return rows;
}

// ── Aggregate by period ───────────────────────────────────────────────────────
function buildPeriod(rows, mode) {
  // For annual mode, vigentes = stock at end of year (last month), not sum
  // We need to track the max-month per year for stock fields
  const map = {};
  const lastMonth = {}; // key -> franquicia -> latest MES seen

  for (const r of rows) {
    const key = mode === "anual" ? r.MES.slice(0,4) : r.MES;
    if (!map[key]) map[key] = {};
    const f = r.FRANQUICIA;
    if (!map[key][f]) map[key][f] = {
      mto_nal:0, mto_ext:0, mto_tot:0,
      num_nal:0, num_ext:0,
      vigentes:0, canceladas:0, vigentes_mes:0,
      saldo:0, cupo:0,
      int_corr:0, int_mora:0, castigos:0,
    };
    const d = map[key][f];
    d.mto_nal   += (r.MTO_COMPRAS_NAL    || 0);
    d.mto_ext   += (r.MTO_COMPRAS_EXT    || 0);
    d.mto_tot   += (r.MTO_COMPRAS_CREDITO|| 0);
    d.num_nal   += (r.NUM_COMPRAS_NAL    || 0);
    d.num_ext   += (r.NUM_COMPRAS_EXT    || 0);
    d.canceladas += (r.CANCELADAS        || 0);
    d.vigentes_mes+=(r.VIGENTES_MES      || 0);
    d.saldo     += (r.SALDO_CARTERA      || 0);
    d.cupo      += (r.CUPO_NO_UTILIZADO  || 0);
    d.int_corr  += (r.INTERESES_CORRIENTES||0);
    d.int_mora  += (r.INTERESES_MORA     || 0);
    d.castigos  += (r.CASTIGOS_TOTAL     || 0);
    // For vigentes: keep last-month value (stock, not flow)
    if (mode === "anual") {
      if (!lastMonth[key]) lastMonth[key] = {};
      if (!lastMonth[key][f] || r.MES > lastMonth[key][f].mes) {
        lastMonth[key][f] = { mes: r.MES, vigentes: r.VIGENTES_FECHA_CORTE||0 };
      }
    } else {
      d.vigentes += (r.VIGENTES_FECHA_CORTE||0);
    }
  }
  // Apply annual vigentes from last month
  if (mode === "anual") {
    for (const key of Object.keys(lastMonth)) {
      for (const f of Object.keys(lastMonth[key])) {
        if (map[key]?.[f]) map[key][f].vigentes = lastMonth[key][f].vigentes;
      }
    }
  }
  const periods = Object.keys(map).sort();
  for (const p of periods) {
    const total = ALL_F.reduce((s,f) => s + (map[p][f]?.mto_tot||0), 0);
    for (const f of ALL_F) {
      if (map[p][f]) map[p][f].ms = total > 0 ? map[p][f].mto_tot / total : 0;
    }
  }
  return {map, periods};
}

// ── Controls ──────────────────────────────────────────────────────────────────
function setTipo(tipo, el) {
  currentTipo = tipo;
  RAW = tipo === "credito" ? RAW_CREDITO
      : tipo === "debito"  ? RAW_DEBITO
      :                      RAW_TOTAL;
  // Total uses ALL_F_CRED so Amex/Diners are included in MS denominator
  // Débito uses only VISA+MC since those are the only franquicias in RAW_DEBITO
  ALL_F = (tipo === "credito" || tipo === "total") ? ALL_F_CRED : ["VISA","MASTERCARD"];
  document.querySelectorAll("#tipo-btn-cred,#tipo-btn-deb,#tipo-btn-tot")
    .forEach(b => b.classList.remove("active"));
  el.classList.add("active");
  populateBancos();
  populateFranqs();
  const note = document.getElementById("filter-note");
  if (note) note.textContent =
      tipo === "debito" ? "Débito: MS via TII (proxy) · Nequi separado · DaviPlata incluida en Davivienda"
    : tipo === "total"  ? "Total: crédito (V+MC+Amex+Diners) + Débito (V+MC) · MS sobre mercado completo"
    : "";
  // Update subtitles
  const volSub = document.getElementById("vol-sub");
  if (volSub) volSub.textContent = tipo !== "credito"
    ? "Monto compras débito por franquicia (COP)"
    : "Monto compras nacionales + exterior por franquicia (COP)";
  const txnSub = document.getElementById("txn-sub");
  if (txnSub) txnSub.textContent = tipo !== "credito"
    ? "Compras débito · nacional + internacional"
    : "Compras nacionales + exterior";
  const vigSub = document.getElementById("vig-sub");
  if (vigSub) vigSub.textContent = tipo !== "credito"
    ? "Tarjetas débito vigentes al corte"
    : "Total tarjetas vigentes al corte del período";
  const yoySub = document.getElementById("yoy-sub");
  if (yoySub) yoySub.textContent =
      tipo === "total"  ? "Variación anual · Visa vs Mastercard · crédito + débito combinado"
    : tipo === "debito" ? "Variación anual · Visa vs Mastercard · débito (proxy TII)"
    :                     "Variación anual · Visa vs Mastercard · últimos 18 meses";
  renderAll();
}

function setView(v, el) {
  currentView = v;
  // Sync the filter-bar period buttons (don't touch vol tabs)
  ["mes","ano"].forEach(s => document.getElementById("period-btn-"+s)?.classList.remove("active"));
  document.getElementById("period-btn-"+(v==="mensual"?"mes":"ano"))?.classList.add("active");
  el.classList.add("active");
  // re-activate vol tab
  document.getElementById("vol-tab-"+volTab)?.classList.add("active");
  renderAll();
}
function setVolTab(t, el) {
  volTab = t;
  ["tot","nal","ext"].forEach(v => document.getElementById("vol-tab-"+v)?.classList.remove("active"));
  el.classList.add("active");
  const {map, periods} = buildPeriod(filtered(), volPeriod);
  renderVolChart(map, periods);
}
function setVolPeriod(p, el) {
  volPeriod = p;
  ["mes","ano"].forEach(v => document.getElementById("vol-period-"+v)?.classList.remove("active"));
  el.classList.add("active");
  const {map, periods} = buildPeriod(filtered(), volPeriod);
  renderVolChart(map, periods);
}

// ── Master render ─────────────────────────────────────────────────────────────

// ── Mapa de Calor Banco × Mes ─────────────────────────────────────────────────
function renderHeatmap(rows) {
  const isAnual = currentView === "anual";
  const displayPeriods = isAnual
    ? [...new Set(rows.map(r => r.MES.slice(0,4)))].sort()
    : [...new Set(rows.map(r => r.MES))].sort();
  const displayMonths = displayPeriods;
  if (!displayMonths.length) return;

  const bs_hm = getPillVals("banco");
  const rowsBanco = bs_hm ? RAW.filter(r=>bs_hm.includes(r.ENTIDAD)) : RAW;

  // Build per-bank per-period aggregated by year or month
  const allData = {}, mktByMonth = {};
  for (const r of rowsBanco) {
    if (!ALL_F.includes(r.FRANQUICIA)) continue;
    const periodKey = isAnual ? r.MES.slice(0,4) : r.MES;
    mktByMonth[periodKey] = (mktByMonth[periodKey]||0) + (r.MTO_COMPRAS_CREDITO||0);
    if (r.FRANQUICIA !== heatmapFranq) continue;
    if (!allData[r.ENTIDAD]) allData[r.ENTIDAD] = {};
    allData[r.ENTIDAD][periodKey] = (allData[r.ENTIDAD][periodKey]||0) + (r.MTO_COMPRAS_CREDITO||0);
  }

  // Banks present in filtered rows, sorted by volume in last displayed month
  const lastM = displayPeriods[displayPeriods.length-1];
  const bancos = [...new Set(rows.map(r => r.ENTIDAD))]
    .filter(b => allData[b])
    .sort((a,b) => (allData[b]?.[lastM]||0) - (allData[a]?.[lastM]||0))
    .slice(0, 15);

  let tableHTML = `<table style="border-collapse:collapse;font-size:9px;width:100%;border-spacing:0">`;
  tableHTML += `<tr><th style="text-align:left;padding:3px 6px;min-width:120px;font-size:10px;color:var(--muted);position:sticky;left:0;z-index:2;background:var(--card-bg,#fff)">Banco</th>`;
  for (const mo of displayMonths) {
    tableHTML += `<th style="text-align:center;padding:2px;color:var(--muted);white-space:nowrap">${mo.slice(0,7)}</th>`;
  }
  tableHTML += `</tr>`;

  // ── Fila 1: Market Share total de la franquicia por mes ─────────────────────
  const franqLabel = heatmapFranq === "VISA" ? "Visa" : "Mastercard";
  const stickyCell = `position:sticky;left:0;z-index:1;background:var(--card-bg,#fff);border-right:1px solid #E2E8F0`;

  tableHTML += `<tr>`;
  tableHTML += `<td style="padding:3px 6px;font-size:10px;font-weight:700;white-space:nowrap;color:var(--text);${stickyCell}">MS ${franqLabel}</td>`;
  for (const mo of displayMonths) {
    const franqVol = Object.values(allData).reduce((s, bData) => s + (bData[mo]||0), 0);
    const ms = mktByMonth[mo] > 0 ? franqVol / mktByMonth[mo] * 100 : null;
    if (ms === null) {
      tableHTML += `<td style="text-align:center;padding:2px;font-size:9px;color:var(--faint)">—</td>`;
    } else {
      tableHTML += `<td style="text-align:center;padding:2px;font-size:10px;font-weight:700;color:var(--text)">${ms.toFixed(1)}%</td>`;
    }
  }
  tableHTML += `</tr>`;

  // ── Fila 2: Δ MS neto de la franquicia (suma de aportes de todos los bancos) ──
  // = MS franquicia este mes - MS franquicia mismo mes año anterior
  tableHTML += `<tr style="border-bottom:2px solid #E2E8F0">`;
  tableHTML += `<td style="padding:3px 6px;font-size:10px;font-weight:700;white-space:nowrap;color:var(--text);${stickyCell}">Δ MS ${franqLabel}</td>`;
  for (const mo of displayMonths) {
    const priorMo = isAnual ? String(parseInt(mo)-1) : (() => { const d = new Date(mo+"-01"); d.setMonth(d.getMonth()-12); return d.toISOString().slice(0,7); })();
    const franqVolNow   = Object.values(allData).reduce((s, bData) => s + (bData[mo]||0), 0);
    const franqVolPrior = Object.values(allData).reduce((s, bData) => s + (bData[priorMo]||0), 0);
    const msNow   = mktByMonth[mo]      > 0 ? franqVolNow   / mktByMonth[mo]      * 100 : null;
    const msPrior = mktByMonth[priorMo] > 0 ? franqVolPrior / mktByMonth[priorMo] * 100 : null;
    const delta = (msNow !== null && msPrior !== null) ? msNow - msPrior : null;
    if (delta === null) {
      tableHTML += `<td style="text-align:center;padding:2px;font-size:9px;color:var(--faint)">—</td>`;
    } else {
      const col = delta >= 0 ? "#15803D" : "#DC2626";
      const sign = delta >= 0 ? "+" : "";
      tableHTML += `<td style="text-align:center;padding:2px;font-size:9px;font-weight:700;color:${col}">${sign}${delta.toFixed(2)}pp</td>`;
    }
  }
  tableHTML += `</tr>`;

  for (const bank of bancos) {
    tableHTML += `<tr><td style="padding:3px 6px;font-size:10px;white-space:nowrap;color:var(--text);position:sticky;left:0;z-index:1;background:var(--card-bg,#fff);border-right:1px solid #E2E8F0">${fmtBank(bank)}</td>`;
    for (const mo of displayMonths) {
      const prior = isAnual ? String(parseInt(mo)-1) : (() => { const d = new Date(mo+"-01"); d.setMonth(d.getMonth()-12); return d.toISOString().slice(0,7); })();
      const msC = mktByMonth[mo]>0   ? (allData[bank]?.[mo]   ||0)/mktByMonth[mo]   *100 : null;
      const msP = mktByMonth[prior]>0 ? (allData[bank]?.[prior]||0)/mktByMonth[prior]*100 : null;
      const diff = (msC!=null && msP!=null) ? msC - msP : null;
      if (diff === null) {
        tableHTML += `<td style="text-align:center;padding:2px;color:var(--faint)">—</td>`;
      } else {
        const intensity = Math.min(Math.abs(diff) / 1.5, 1);
        const r = diff >= 0 ? Math.round(240 - intensity*180) : 220;
        const g = diff >= 0 ? 220 : Math.round(240 - intensity*180);
        const b2 = Math.round(240 - intensity*180);
        const bg = `rgba(${r},${g},${b2},${0.15 + intensity*0.65})`;
        const textCol = intensity > 0.5 ? '#fff' : 'var(--text)';
        tableHTML += `<td style="text-align:center;padding:2px;background:${bg};color:${textCol};border-radius:3px" title="${diff>0?'+':''}${diff.toFixed(2)}pp">${diff>0?'+':''}${diff.toFixed(1)}</td>`;
      }
    }
    tableHTML += `</tr>`;
  }
  tableHTML += `</table>`;
  const container = document.getElementById("heatmap-container");
  if (container) container.innerHTML = tableHTML;
}

// ── Retiros ATM ──────────────────────────────────────────────────────────────
function renderRetiros() {
  // Same logic as renderCartera — respects currentView, viewPeriods, año/mes filters
  const bs = getPillVals("banco");
  const rows    = bs ? RAW_RETIROS.filter(r => bs.includes(r.ENTIDAD)) : RAW_RETIROS;
  const rowsMkt = RAW_RETIROS; // market reference always unfiltered by banco

  if (!rows || !rows.length) return;

  // Build period map (mensual or anual) same as buildPeriod
  const allMes = [...new Set(RAW_RETIROS.map(r => r.MES))].sort();
  // Aggregate by period key
  function buildRatioMap(rowsSrc, mode) {
    const map = {};
    for (const r of rowsSrc) {
      const key = mode === "anual" ? r.MES.slice(0,4) : r.MES;
      if (!map[key]) map[key] = {ret_v:0, comp_v:0, ret_m:0, comp_m:0, ret_all:0, comp_all:0};
      const d = map[key];
      if (r.FRANQUICIA === "VISA")       { d.ret_v += r.MTO_RETIROS||0; d.comp_v += r.MTO_COMPRAS||0; }
      if (r.FRANQUICIA === "MASTERCARD") { d.ret_m += r.MTO_RETIROS||0; d.comp_m += r.MTO_COMPRAS||0; }
      d.ret_all  += r.MTO_RETIROS||0;
      d.comp_all += r.MTO_COMPRAS||0;
    }
    return map;
  }

  const mode     = currentView;
  const mapRows  = buildRatioMap(rows, mode);
  const mapMkt   = buildRatioMap(rowsMkt, mode);
  const allPeriods = [...new Set([...Object.keys(mapRows), ...Object.keys(mapMkt)])].sort();

  // Apply viewPeriods logic (año/mes filter + last 24 fallback)
  const years = getPillVals("year");
  const meses = getPillVals("mes");
  let displayP = allPeriods;
  if (years) displayP = displayP.filter(p => years.includes(p.slice(0,4)));
  if (meses && mode !== "anual") displayP = displayP.filter(p => meses.includes(p.slice(5,7)));
  if (!years && !meses) displayP = displayP.slice(-24);

  const labels   = displayP;
  const ratio = (d, num, den) => d[den] > 0 ? +(d[num]/d[den]*100).toFixed(2) : null;
  const visaLine = displayP.map(p => { const d = mapRows[p]; return d ? ratio(d,'ret_v','comp_v'+(d.comp_v>0?'':'')||1) : null; });
  // fix: proper ratio
  const visaL = displayP.map(p => { const d=mapRows[p]; return d&&(d.ret_v+d.comp_v)>0 ? +((d.ret_v/(d.ret_v+d.comp_v))*100).toFixed(2) : null; });
  const mcL   = displayP.map(p => { const d=mapRows[p]; return d&&(d.ret_m+d.comp_m)>0 ? +((d.ret_m/(d.ret_m+d.comp_m))*100).toFixed(2) : null; });
  const mktL  = displayP.map(p => { const d=mapMkt[p];  return d&&(d.ret_all+d.comp_all)>0 ? +((d.ret_all/(d.ret_all+d.comp_all))*100).toFixed(2) : null; });

  mkChart("retirosChart", {
    type: "line",
    data: { labels, datasets: [
      { label:"Visa",          data:visaL, borderColor:VISA,      backgroundColor:"transparent",
        pointRadius:2, tension:0.2, borderWidth:2, spanGaps:true },
      { label:"Mastercard",    data:mcL,   borderColor:MC,        backgroundColor:"transparent",
        pointRadius:2, tension:0.2, borderWidth:2, spanGaps:true },
      { label:"Total mercado", data:mktL,  borderColor:"#94A3B8", backgroundColor:"transparent",
        pointRadius:0, tension:0.2, borderWidth:1.5, borderDash:[5,4], spanGaps:true },
    ]},
    options: {
      responsive:true, maintainAspectRatio:false,
      layout:{ padding:{top:10,right:10,bottom:2,left:2} },
      interaction:{ mode:"index", intersect:false },
      plugins:{
        legend:{ display:true, position:"bottom",
          labels:{ boxWidth:12, font:{size:10}, padding:8 }},
        tooltip:{ mode:"index", intersect:false,
          callbacks:{ label: c => `${c.dataset.label}: ${c.parsed.y!=null?c.parsed.y.toFixed(1)+"%":"—"}` }}
      },
      scales:{
        x:{ ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:mode==="anual"?15:20,font:{size:9}}, grid:{display:false} },
        y:{ grace:"5%", ticks:{callback:v=>v.toFixed(0)+"%",font:{size:9}}, grid:{color:"#F2F4F7"} }
      }
    }
  });
}

function setHeatmapFranq(f, el) {
  heatmapFranq = f;
  document.querySelectorAll("#heatmap-franq-tabs .tab").forEach(b => b.classList.remove("active"));
  el.classList.add("active");
  renderHeatmap(filtered());
}

// ── Análisis de Cohorte ────────────────────────────────────────────────────────
function renderCohort(rows) {
  // ultimo = last month in filtered rows; for YoY use banco-only RAW
  const filteredMonths = [...new Set(rows.map(r => r.MES))].sort();
  if (!filteredMonths.length) return;
  const ultimo = filteredMonths[filteredMonths.length-1];

  const bs_co = getPillVals("banco");
  const rowsBanco = bs_co ? RAW.filter(r=>bs_co.includes(r.ENTIDAD)) : RAW;
  const allMonths = [...new Set(rowsBanco.map(r => r.MES))].sort();
  const uIdx = allMonths.indexOf(ultimo);
  const prior = uIdx >= 12 ? allMonths[uIdx-12] : allMonths[0];

  // Total market (all franquicias) at ultimo and prior
  const mktU = {}, mktP = {};
  const activeFranqs = [...new Set(rowsBanco.map(r=>r.FRANQUICIA))];
  for (const r of rowsBanco) {
    if (!activeFranqs.includes(r.FRANQUICIA)) continue;
    if (r.MES === ultimo) mktU.tot = (mktU.tot||0) + (r.MTO_COMPRAS_CREDITO||0);
    if (r.MES === prior)  mktP.tot = (mktP.tot||0) + (r.MTO_COMPRAS_CREDITO||0);
  }

  // Per-bank facturación for the selected franquicia at ultimo and prior
  // Use rows (filtered) for ultimo, rowsBanco for prior
  const bankTotU = {}, bankTotP = {};
  for (const r of rows) {
    if (r.FRANQUICIA !== cohortFranq || r.MES !== ultimo) continue;
    bankTotU[r.ENTIDAD] = (bankTotU[r.ENTIDAD]||0) + (r.MTO_COMPRAS_CREDITO||0);
  }
  for (const r of rowsBanco) {
    if (r.FRANQUICIA !== cohortFranq || r.MES !== prior) continue;
    bankTotP[r.ENTIDAD] = (bankTotP[r.ENTIDAD]||0) + (r.MTO_COMPRAS_CREDITO||0);
  }

  const totalMktU = mktU.tot||0, totalMktP = mktP.tot||0;
  const points = [];
  for (const bank of Object.keys(bankTotU)) {
    const valU = bankTotU[bank]||0, valP = bankTotP[bank]||0;
    if (valU < 1e9) continue;
    const msU = totalMktU>0 ? valU/totalMktU*100 : 0;
    const yoy = valP>0 ? (valU/valP-1)*100 : null;
    if (yoy===null) continue;
    points.push({ bank: fmtBank(bank), ms: msU, yoy, vol: valU });
  }
  if (!points.length) return;

  const maxVol = Math.max(...points.map(p=>p.vol));
  const color  = cohortFranq === "VISA" ? VISA : MC;
  const avgMs  = points.reduce((s,p)=>s+p.ms,0)/points.length;
  const avgYoy = 0; // fixed at 0% so quadrants split growth vs decline

  mkChart("cohortChart", {
    type: "bubble",
    data: { datasets: points.map(p => ({
      label: p.bank,
      data: [{ x: p.ms, y: p.yoy, r: Math.max(6, Math.sqrt(p.vol/maxVol)*32) }],
      backgroundColor: color + "99",
      borderColor: color,
      borderWidth: 1.5,
    }))},
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        tooltip: { callbacks: { label: c => {
          const p = points[c.datasetIndex];
          return [`${p.bank}`, `MS: ${p.ms.toFixed(2)}%`, `YoY: ${p.yoy>0?"+":""}${p.yoy.toFixed(1)}%`, `Vol: ${fmtB(p.vol)}`];
        }}}
      },
      scales: {
        x: { title:{display:true,text:"Market Share (%) — "+cohortFranq+" · "+ultimo,font:{size:10}}, ticks:{callback:v=>v.toFixed(1)+"%",font:{size:9}}, grid:{color:"#F2F4F7"} },
        y: { title:{display:true,text:"Crecimiento YoY vs "+prior,font:{size:10}}, ticks:{callback:v=>(v>0?"+":"")+v.toFixed(0)+"%",font:{size:9}}, grid:{color:"#F2F4F7"} }
      },
      animation: { duration: 400 }
    },
    plugins: [{ id:"quadrantLines", afterDraw(chart) {
      const {ctx, chartArea:{left,right,top,bottom}, scales:{x,y}} = chart;
      const cx = x.getPixelForValue(avgMs), cy = y.getPixelForValue(avgYoy);
      ctx.save();
      ctx.strokeStyle="#CBD5E1"; ctx.lineWidth=1; ctx.setLineDash([4,4]);
      ctx.beginPath(); ctx.moveTo(cx,top); ctx.lineTo(cx,bottom); ctx.stroke();
      ctx.beginPath(); ctx.moveTo(left,cy); ctx.lineTo(right,cy); ctx.stroke();
      ctx.setLineDash([]);
      ctx.font="9px sans-serif"; ctx.fillStyle="#94A3B8";
      ctx.fillText("Alto MS / Creciendo", cx+6, top+14);
      ctx.fillText("Bajo MS / Creciendo", left+6, top+14);
      ctx.fillText("Alto MS / Perdiendo", cx+6, bottom-6);
      ctx.fillText("Bajo MS / Perdiendo", left+6, bottom-6);
      ctx.font="9px sans-serif"; ctx.fillStyle="#334155";
      chart.data.datasets.forEach((ds,i) => {
        const meta = chart.getDatasetMeta(i);
        if (meta.hidden || !meta.data[0]) return;
        const el = meta.data[0];
        ctx.fillText(ds.label, el.x+el.options.radius+2, el.y+3);
      });
      ctx.restore();
    }}]
  });
}

function setCohortFranq(f, el) {
  cohortFranq = f;
  document.querySelectorAll("#cohort-franq-tabs .tab").forEach(b => b.classList.remove("active"));
  el.classList.add("active");
  renderCohort(filtered());
}

function renderAll() {
  const rows = filtered();
  const {map, periods} = buildPeriod(rows, currentView);
  const ultimo  = periods[periods.length-1];
  const penult  = periods[periods.length-2] || periods[0];
  const tipoLabel = currentTipo === "debito" ? " — Débito"
                 : currentTipo === "total"  ? " — Total (Crédito + Débito)"
                 : "";
  document.getElementById("ms-title").textContent =
    (currentView === "anual" ? "Evolución anual del market share" : "Evolución mensual del market share") + tipoLabel;

  // yoyMap: full monthly history, only banco-filtered — used for YoY deltas everywhere
  const rowsBanco = (() => { const bs=getPillVals("banco"); return bs?RAW.filter(r=>bs.includes(r.ENTIDAD)):RAW; })();
  const {map:yoyMap, periods:yoyPeriods} = buildPeriod(rowsBanco, "mensual");
  const yoyUltimo = yoyPeriods[yoyPeriods.length-1];
  // yoyPrev: same month 12 periods back in full monthly map
  const yoyPrevKey = yoyPeriods.length >= 13 ? yoyPeriods[yoyPeriods.length-13] : yoyPeriods[0];

  renderMSChart(map, periods);

  const {map:mMap, periods:mPeriods} = buildPeriod(rows, "mensual");
  const {map:cMap, periods:cPeriods} = buildPeriod(rows, currentView);
  const {map:vMap, periods:vPeriods} = buildPeriod(rows, volPeriod);
  renderVolChart(vMap, vPeriods);
  renderYoY(rows);
  renderTxnVig(cMap, cPeriods);
  renderContrib(map, ultimo, penult);
  renderIntl(mMap, mPeriods[mPeriods.length-1]);
  renderTicket(cMap, cPeriods);
  renderBancos(rows, mPeriods[mPeriods.length-1]);
  renderCartera(mMap, mPeriods);
  renderContribBancos(rows, ultimo, penult);
  renderSnap(map, periods, yoyMap);
  renderRetiros();
  renderHeatmap(rows);
  renderCohort(rows);
}

// ── VIZ 1 — 100% stacked bar (horizontal time) ───────────────────────────────
function renderMSChart(map, periods) {
  const labels = periods.map(p => p.length===7 ? p.slice(0,7) : p);
  const datasets = ALL_F.map(f => ({
    label: f,
    data: periods.map(p => +((map[p]?.[f]?.ms||0)*100).toFixed(2)),
    backgroundColor: C[f],
    borderWidth: 0,
    categoryPercentage: 0.85,
    barPercentage: 1.0,
  }));
  mkChart("msChart", {
    type: "bar",
    data: {labels, datasets},
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: {display:false},
        tooltip: {
          mode:"index", intersect:false,
          callbacks: { label: c => `${c.dataset.label}: ${c.parsed.y.toFixed(1)}%` }
        }
      },
      scales: {
        x: { stacked:true, ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}}, grid:{display:false} },
        y: { stacked:true, min:0, max:100, ticks:{callback:v=>v+"%",font:{size:10}}, grid:{color:"#F2F4F7"} }
      }
    }
  });
}

// ── VIZ 2/3 — Facturación con 3 tabs ─────────────────────────────────────────
function renderVolChart(map, periods) {
  const last = volPeriod === "anual" ? periods : periods.slice(-24);
  const labels = volPeriod === "anual" ? last : last.map(p => p.slice(5));
  const key = volTab === "tot" ? "mto_tot" : volTab === "nal" ? "mto_nal" : "mto_ext";
  const rawV = last.map(p => map[p]?.VISA?.[key]||0);
  const rawM = last.map(p => map[p]?.MASTERCARD?.[key]||0);
  const sc = chartScale([...rawV, ...rawM]);
  mkChart("volChart", {
    type: "bar",
    data: { labels, datasets: [
      {label:"Visa",       data:rawV.map(v=>+(v/sc.div).toFixed(2)), backgroundColor:VISA+"CC", borderWidth:0},
      {label:"Mastercard", data:rawM.map(v=>+(v/sc.div).toFixed(2)), backgroundColor:MC+"CC",   borderWidth:0},
    ]},
    options: {
      responsive:true, maintainAspectRatio:false,
      plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>`${c.dataset.label}: $${c.parsed.y.toFixed(2)} ${sc.sfx} COP`}}},
      scales:{
        x:{ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}},grid:{display:false}},
        y:{ticks:{callback:v=>"$"+v.toFixed(1)+sc.sfx, font:{size:9}},grid:{color:"#F2F4F7"}}
      }
    }
  });
}

// ── VIZ YoY ───────────────────────────────────────────────────────────────────
function setYoyTab(t, el) {
  yoyTab = t;
  ["fact","txn","vig","tkt"].forEach(v => document.getElementById("yoy-tab-"+v)?.classList.remove("active"));
  el.classList.add("active");
  const labels = {"fact":"Facturación","txn":"Transacciones","vig":"Tarjetas Vigentes","tkt":"Ticket Promedio"};
  document.getElementById("yoy-title").textContent = "Crecimiento YoY — " + labels[t];
  renderYoY(filtered());
}

function renderYoY(rows) {
  // In total mode: use RAW_TOTAL (cred+deb combined) for YoY
  // In debito mode: use RAW_DEBITO
  // In credito mode: use RAW_CREDITO (cleanest signal)
  const rawSrc = currentTipo === "total"  ? RAW_TOTAL
               : currentTipo === "debito" ? RAW_DEBITO
               :                           RAW_CREDITO;
  const bs = getPillVals("banco");
  const rowsBanco = bs ? rawSrc.filter(r=>bs.includes(r.ENTIDAD)) : rawSrc;
  const {map:fullMap, periods:fullPeriods} = buildPeriod(rowsBanco, "mensual");

  // Which months to show: apply año+mes filters
  const years = getPillVals("year");
  const meses  = getPillVals("mes");
  let displayPeriods = fullPeriods;
  if (years) displayPeriods = displayPeriods.filter(p => years.includes(p.slice(0,4)));
  if (meses)  displayPeriods = displayPeriods.filter(p => meses.includes(p.slice(5,7)));
  // Need at least 12 months of history — always use fullMap for lookups
  const labels=[], vY=[], mY=[], totY=[];
  const keyFn = yoyTab === "fact" ? d => d.mto_tot
               : yoyTab === "txn"  ? d => (d.num_nal||0)+(d.num_ext||0)
               : yoyTab === "tkt"  ? d => d.num_nal>0 ? d.mto_nal/d.num_nal : null
               :                     d => d.vigentes||0;
  for (const p of displayPeriods) {
    // prior = same month 12 periods back in fullPeriods
    const pIdx = fullPeriods.indexOf(p);
    if (pIdx < 12) continue;
    const p12 = fullPeriods[pIdx-12];
    const vN=keyFn(fullMap[p]?.VISA||{}),  vO=keyFn(fullMap[p12]?.VISA||{});
    const mN=keyFn(fullMap[p]?.MASTERCARD||{}), mO=keyFn(fullMap[p12]?.MASTERCARD||{});
    // Use ALL_F (dynamic per tipo) — matches snapshot formula: sum all active franquicias
    const activeFranqs = Object.keys(fullMap[p]||{});
    const totN = activeFranqs.reduce((s,f) => { const v=keyFn(fullMap[p]?.[f]||{}); return s+(v||0); }, 0);
    const totO = activeFranqs.reduce((s,f) => { const v=keyFn(fullMap[p12]?.[f]||{}); return s+(v||0); }, 0);
    if (vO>0 && mO>0) {
      labels.push(p.slice(0,7));
      vY.push((vN/vO-1)*100);
      mY.push((mN/mO-1)*100);
      totY.push(totO>0 ? (totN/totO-1)*100 : null);
    }
  }
  // If no year/mes filter, show last 18 months
  const last  = (!years && !meses) ? labels.slice(-18) : labels;
  const vD    = (!years && !meses) ? vY.slice(-18)     : vY;
  const mD    = (!years && !meses) ? mY.slice(-18)     : mY;
  const tD    = (!years && !meses) ? totY.slice(-18)   : totY;
  mkChart("yoyChart", {
    type:"line",
    data:{labels:last, datasets:[
      {label:"Visa",         data:vD, borderColor:VISA,     backgroundColor:"transparent", pointRadius:2, tension:0.3, borderWidth:2},
      {label:"Mastercard",   data:mD, borderColor:MC,       backgroundColor:"transparent", pointRadius:2, tension:0.3, borderWidth:2},
      {label:"Total mercado",data:tD, borderColor:"#94A3B8",backgroundColor:"transparent", pointRadius:0, tension:0.3, borderWidth:1.5, borderDash:[5,4]},
    ]},
    options: lineOpts("%")
  });
}

// ── Transacciones + Tarjetas vigentes ────────────────────────────────────────
function renderTxnVig(map, periods) {
  const last = viewPeriods(periods);
  const labels = viewLabels(periods);

  // ── helper: YoY label for latest point ──
  function yoyLabel(vals) {
    if (vals.length < 13) return null;
    const cur = vals[vals.length-1], prev = vals[vals.length-13];
    if (!prev || !cur) return null;
    return (cur/prev - 1) * 100;
  }
  function yoyHtml(v, prefix) {
    if (v == null) return "";
    const cls = v > 0 ? "chip-up" : v < 0 ? "chip-down" : "chip-neu";
    return `${prefix}: <span class="chip ${cls}">${v>0?"+":""}${v.toFixed(1)}%</span>`;
  }

  // Transactions
  const rawVT = last.map(p => (map[p]?.VISA?.num_nal||0) + (map[p]?.VISA?.num_ext||0));
  const rawMT = last.map(p => (map[p]?.MASTERCARD?.num_nal||0) + (map[p]?.MASTERCARD?.num_ext||0));
  const scT = chartScale([...rawVT, ...rawMT]);
  mkChart("txnChart", {
    type: "line",
    data: { labels, datasets: [
      {label:"Visa",       data:rawVT.map(v=>+(v/scT.div).toFixed(2)), borderColor:VISA, backgroundColor:VISA+"18", fill:true, pointRadius:0, tension:0.3, borderWidth:2},
      {label:"Mastercard", data:rawMT.map(v=>+(v/scT.div).toFixed(2)), borderColor:MC,   backgroundColor:MC+"18",   fill:true, pointRadius:0, tension:0.3, borderWidth:2},
    ]},
    options: {
      responsive:true, maintainAspectRatio:false,
      layout:{padding:{top:10,right:10,bottom:2,left:2}},
      interaction:{mode:"index",intersect:false},
      plugins:{legend:{display:false},tooltip:{mode:"index",intersect:false,callbacks:{
        label: c => `${c.dataset.label}: ${c.parsed.y.toFixed(2)} ${scT.sfx} txn`
      }}},
      scales:{
        x:{ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}},grid:{display:false}},
        y:{grace:"8%",ticks:{callback:v=>v.toFixed(1)+scT.sfx,font:{size:9}},grid:{color:"#F2F4F7"}}
      }
    }
  });
  const yoyV = yoyLabel(rawVT), yoyM = yoyLabel(rawMT);
  document.getElementById("txn-yoy-visa").innerHTML = yoyHtml(yoyV, "Visa YoY");
  document.getElementById("txn-yoy-mc").innerHTML   = yoyHtml(yoyM, "MC YoY");

  // Tarjetas vigentes — snapshot del último mes del período (no acumulado)
  const rawVV = last.map(p => map[p]?.VISA?.vigentes||0);
  const rawMV = last.map(p => map[p]?.MASTERCARD?.vigentes||0);
  const scV = chartScale([...rawVV, ...rawMV]);
  mkChart("vigChart", {
    type: "line",
    data: { labels, datasets: [
      {label:"Visa",       data:rawVV.map(v=>+(v/scV.div).toFixed(2)), borderColor:VISA, backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:2},
      {label:"Mastercard", data:rawMV.map(v=>+(v/scV.div).toFixed(2)), borderColor:MC,   backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:2},
    ]},
    options: {
      responsive:true, maintainAspectRatio:false,
      layout:{padding:{top:10,right:10,bottom:2,left:2}},
      interaction:{mode:"index",intersect:false},
      plugins:{legend:{display:false},tooltip:{mode:"index",intersect:false,callbacks:{
        label: c => `${c.dataset.label}: ${c.parsed.y.toFixed(2)} ${scV.sfx}`
      }}},
      scales:{
        x:{ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}},grid:{display:false}},
        y:{grace:"8%",ticks:{callback:v=>v.toFixed(1)+scV.sfx,font:{size:9}},grid:{color:"#F2F4F7"}}
      }
    }
  });
  const yoyVV = yoyLabel(rawVV), yoyMV = yoyLabel(rawMV);
  document.getElementById("vig-yoy-visa").innerHTML = yoyHtml(yoyVV, "Visa YoY");
  document.getElementById("vig-yoy-mc").innerHTML   = yoyHtml(yoyMV, "MC YoY");
}

// ── Contribución ──────────────────────────────────────────────────────────────
function renderContrib(map, ultimo, penult) {
  let html = "";
  for (const f of ALL_F) {
    const curr = (map[ultimo]?.[f]?.ms||0)*100;
    const prev = (map[penult]?.[f]?.ms||0)*100;
    const diff = curr - prev;
    const bw = Math.min(Math.abs(diff)*9, 100);
    const bc = diff >= 0 ? C[f] : "#D0D5DD";
    const ch = Math.abs(diff)<0.05
      ? `<span class="chip chip-neu">~0pp</span>`
      : diff>0
        ? `<span class="chip chip-up">+${diff.toFixed(1)}pp</span>`
        : `<span class="chip chip-down">${diff.toFixed(1)}pp</span>`;
    html += `<div style="margin-bottom:11px">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:3px">
        <span style="font-size:11px;color:var(--muted)">${f}</span>
        <span style="font-size:11px">${ch} <strong>${curr.toFixed(1)}%</strong></span>
      </div>
      <div class="bar-bg"><div class="bar-fill" style="width:${bw}%;background:${bc}"></div></div>
    </div>`;
  }
  document.getElementById("contrib-body").innerHTML = html;
  document.getElementById("contrib-sub").textContent = `Δ pp vs ${penult}`;
}

// ── Nacional vs Exterior ──────────────────────────────────────────────────────
function renderIntl(map, ultimo) {
  // In débito/total mode: SFC doesn't separate nacional vs exterior for débito
  // Show a note instead of the chart
  const intlBody = document.getElementById("intl-body");
  if (!intlBody) return;
  if (currentTipo !== "credito") {
    intlBody.innerHTML = '<div style="color:var(--muted);font-size:12px;padding:8px 0">' +
      'La SFC no desagrega nacional vs exterior para tarjetas débito. ' +
      'El monto reportado incluye ambos circuitos.</div>';
    return;
  }
  let html = "";
  for (const f of ["VISA","MASTERCARD"]) {
    const d = map[ultimo]?.[f] || {};
    const nal=d.mto_nal||0, ext=d.mto_ext||0, tot=nal+ext;
    if (!tot) continue;
    const pN=(nal/tot*100).toFixed(1), pE=(ext/tot*100).toFixed(1);
    html += `<div style="margin-bottom:16px">
      <div style="display:flex;justify-content:space-between;margin-bottom:5px">
        <span style="font-size:12px;font-weight:700;display:flex;align-items:center;gap:6px">
          <span style="width:8px;height:8px;border-radius:2px;background:${C[f]};display:inline-block"></span>${f}
        </span>
        <span style="font-size:11px;color:var(--muted)">${fmtB(tot)}</span>
      </div>
      <div class="intl-bar">
        <div style="width:${pN}%;background:${C[f]}"></div>
        <div style="flex:1;background:${C[f]}33"></div>
      </div>
      <div style="display:flex;justify-content:space-between;font-size:10px;color:var(--faint);margin-top:3px">
        <span>Nacional ${pN}%</span><span>Exterior ${pE}%</span>
      </div>
    </div>`;
  }
  intlBody.innerHTML = html;
}

// ── Ticket promedio ───────────────────────────────────────────────────────────
function renderTicket(map, periods) {
  const last = viewPeriods(periods);
  const labels = viewLabels(periods);
  // Ticket en miles de pesos COP
  const rawVT = last.map(p => { const d=map[p]?.VISA||{}; return d.num_nal>0 ? Math.round(d.mto_nal/d.num_nal/1000) : null; });
  const rawMT = last.map(p => { const d=map[p]?.MASTERCARD||{}; return d.num_nal>0 ? Math.round(d.mto_nal/d.num_nal/1000) : null; });
  mkChart("ticketChart", {
    type:"line",
    data:{labels, datasets:[
      {label:"Visa",       data:rawVT, borderColor:VISA, backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:2, spanGaps:true},
      {label:"Mastercard", data:rawMT, borderColor:MC,   backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:2, spanGaps:true},
    ]},
    options: {
      responsive:true, maintainAspectRatio:false,
      layout:{padding:{top:10,right:10,bottom:2,left:2}},
      interaction:{mode:"index",intersect:false},
      plugins:{legend:{display:false},tooltip:{mode:"index",intersect:false,callbacks:{label:c=>`${c.dataset.label}: ${c.parsed.y != null ? "$"+c.parsed.y.toLocaleString("es-CO")+"K" : "—"}`}}},
      scales:{
        x:{ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}},grid:{display:false}},
        y:{grace:"8%",ticks:{callback:v=>"$"+Math.round(v).toLocaleString("es-CO")+"K",font:{size:9}},grid:{color:"#F2F4F7"}}
      }
    }
  });
}

// ── Bank name formatter ──────────────────────────────────────────────────────\nfunction fmtBank(raw) {\n  const up = raw.toUpperCase();\n  // Named exceptions\n  if (up.includes("NEQUI"))                  return "Nequi";\
  if (up.includes("BANCOLOMBIA"))            return "Bancolombia";\n  if (up.includes("DE BOGOT"))               return "Banco de Bogotá";\n  if (up.includes("BBVA"))                   return "BBVA Colombia";\n  if (up.includes("CITIBANK"))               return "Citibank";\n  if (up.includes("GNB"))                    return "GNB Sudameris";\n  if (up.includes("AGRARIO"))                return "Agrario de Colombia";\n  if (up.includes("TUYA"))                   return "Tuya";\n  if (up.includes("PROCREDIT"))              return "Procredit";\n  if (up.includes("PICHINCHA"))              return "Pichincha";\n  if (up.includes("SERFINANZA"))             return "Serfinanza";\n  if (up.includes("COOFINEP"))               return "Coofinep";\n  if (up.includes("CONFIAR"))                return "Confiar";
  if (up.includes("SCOTIABANK") || up.includes("COLPATRIA")) return "Scotiabank Colpatria";
  if (up.includes("NU FINANCIERA"))              return "Nu";
  if (up.startsWith("NUBANK"))                   return "Nu";
  if (up.includes("OCCIDENTE"))                  return "Occidente";
  if (up.includes("AV VILLAS"))                  return "Av Villas";
  if (up.includes("JURISCOOP"))                  return "Juriscoop";
  if (up.includes("MUNDO MUJER"))                return "Mundo Mujer";
  if (up.includes("BANCAMIA"))                   return "Bancamia";
  if (up.includes("VISIONAMOS"))                 return "Visionamos";
  if (up.includes("COTRAFA"))                    return "Cotrafa";
  if (up.includes("COOPCENTRAL"))                return "Coopcentral";
  if (up.includes("RAPPIPAY") || up.includes('"RAPPIPAY"')) return "RappiPay";
  if (up.includes("BOLD"))                       return "BOLD";
  if (up.includes("LULO"))                       return "Lulo Bank";
  if (up.includes("FALABELLA"))                  return "Falabella";
  if (up.includes("DAVIVIENDA"))                 return "Davivienda";
  if (up.includes("POPULAR"))                    return "Banco Popular";
  if (up.includes("CAJA SOCIAL"))                return "Caja Social";
  if (up.includes("COOMEVA"))                    return "Coomeva";
  if (up.includes("FINANDINA"))                  return "Finandina";
  if (up.includes("ITAU") || up.includes("CORPBANCA")) return "Itaú";\n  // Generic strip\n  const clean = raw\n    .replace(/BANCO\\s+/i,                   "")\n    .replace(/S\\.A\\.C\\.F\\.?\\s*/ig,       "")\n    .replace(/\\bC\\.F\\b\\s*/ig,            "")\n    .replace(/S\\.A\\.\\s*/ig,              "")\n    .replace(/\\bS\\.A\\b\\s*/ig,            "")\n    .replace(/COMPAÑÍA DE FINANCIAMIENTO/ig, "")\n    .replace(/CORPORACIÓN FINANCIERA/ig,     "")\n    .replace(/COLOMBIA\\s*/ig,               "")\n    .replace(/[-–]\\s*$/,                   "")\n    .replace(/\\s{2,}/g,                    " ")\n    .trim();\n  // Title case\n  return clean.toLowerCase().replace(/(^|\\s)\\S/g, c => c.toUpperCase());\n}\n\n// ── VIZ 10 — Bancos horizontal stacked ───────────────────────────────────────
function renderBancos(rows, ultimo) {
  const data = {};
  for (const r of rows) {
    if (r.MES !== ultimo) continue;
    if (!data[r.ENTIDAD]) data[r.ENTIDAD] = {VISA:0, MASTERCARD:0, "AMERICAN EXPRESS":0, DINERS:0, OTRAS:0};
    if (["VISA","MASTERCARD","AMERICAN EXPRESS","DINERS"].includes(r.FRANQUICIA)) {
      data[r.ENTIDAD][r.FRANQUICIA] += (r.MTO_COMPRAS_CREDITO||0);
    } else {
      // OTRAS TARJETAS DE CREDITO and any other → OTRAS bucket
      data[r.ENTIDAD].OTRAS += (r.MTO_COMPRAS_CREDITO||0);
    }
  }
  const entries = Object.entries(data)
    .map(([e,d]) => [e,
      (d.VISA||0)+(d.MASTERCARD||0)+(d["AMERICAN EXPRESS"]||0)+(d.DINERS||0)+(d.OTRAS||0),
      d.VISA||0, d.MASTERCARD||0, d["AMERICAN EXPRESS"]||0, d.DINERS||0, d.OTRAS||0])
    .filter(e => e[1] > 0)
    .sort((a,b) => b[1]-a[1])
    .slice(0, 20);

  const mktTotal = entries.reduce((s,e) => s+e[1], 0);

  const labels = entries.map(e => fmtBank(e[0]));
  const allVals = entries.flatMap(e => [e[2], e[3], e[4], e[5], e[6]]);
  const sc = chartScale(allVals);
  const vData  = entries.map(e => +(e[2]/sc.div).toFixed(2));
  const mData  = entries.map(e => +(e[3]/sc.div).toFixed(2));
  const aData  = entries.map(e => +(e[4]/sc.div).toFixed(2));
  const dData  = entries.map(e => +(e[5]/sc.div).toFixed(2));
  const oData  = entries.map(e => +(e[6]/sc.div).toFixed(2));

  const h = Math.max(entries.length * 42 + 60, 300);
  document.getElementById("banco-wrap").style.height = h + "px";

  mkChart("bancoChart", {
    type:"bar",
    data:{labels, datasets:[
      {label:"Visa",       data:vData, backgroundColor:VISA+"CC",    borderWidth:0},
      {label:"Mastercard", data:mData, backgroundColor:MC+"CC",      borderWidth:0},
      {label:"Amex",       data:aData, backgroundColor:AMEX+"CC",    borderWidth:0},
      {label:"Diners",     data:dData, backgroundColor:DINERS+"CC",  borderWidth:0},
      {label:"Otras",      data:oData, backgroundColor:"#94A3B8CC",    borderWidth:0},
    ]},
    options:{
      indexAxis:"y",
      responsive:true, maintainAspectRatio:false,
      plugins:{
        legend:{display:false},
        tooltip:{
          callbacks:{
            label: c => {
              const e = entries[c.dataIndex];
              const tot = e[1];
              const fmap = {"Visa":e[2],"Mastercard":e[3],"Amex":e[4],"Diners":e[5],"Otras":e[6]};
              const franqVal = fmap[c.dataset.label] || 0;
              const pctPais   = mktTotal>0 ? (tot/mktTotal*100).toFixed(1) : "—";
              const pctFranq  = tot>0      ? (franqVal/tot*100).toFixed(1)   : "—";
              return [
                `${c.dataset.label}: $${c.parsed.x.toFixed(2)} ${sc.sfx}`,
                `Banco: ${pctPais}% del mercado total`,
                `${c.dataset.label} dentro del banco: ${pctFranq}%`
              ];
            }
          }
        }
      },
      scales:{
        x:{stacked:true, ticks:{callback:v=>"$"+v+sc.sfx,font:{size:9}},grid:{color:"#F2F4F7"}},
        y:{stacked:true, ticks:{font:{size:10}},grid:{display:false}}
      }
    }
  });
}

// ── Cartera: 4 gráficos de salud ─────────────────────────────────────────────
function renderCartera(map, periods) {
  // Respect year/month filters; fall back to last 24 months when no filter active
  const filtered = viewPeriods(periods);
  const last   = filtered.length > 0 ? filtered : periods.slice(-24);
  const labels = last.map(p => p.length === 7 ? p.slice(0,7) : p.slice(0,7));

  // Full market map (banco-filtered RAW, not filtered by año/mes) for market total line
  const rowsBancoC = (() => { const bs=getPillVals("banco"); return bs?RAW.filter(r=>bs.includes(r.ENTIDAD)):RAW; })();
  const {map:fullMapC, periods:fullPeriodsC} = buildPeriod(rowsBancoC, "mensual");
  const lastFull = fullPeriodsC.filter(p => last.includes(p));

  function carteraLine(id, keyFn, fmtStr) {
    const vD  = last.map(p => { const d=map[p]?.VISA||{}; const v=keyFn(d); return (v!=null&&v>0)?+v.toFixed(3):null; });
    const mD  = last.map(p => { const d=map[p]?.MASTERCARD||{}; const v=keyFn(d); return (v!=null&&v>0)?+v.toFixed(3):null; });
    // Market total: aggregate all franquicias
    const totD = last.map(p => {
      const allFranqs = Object.values(fullMapC[p]||{});
      if (!allFranqs.length) return null;
      // For ratio metrics, we need to sum numerator and denominator separately
      // Use weighted average approach: sum(numerator) / sum(denominator)
      let num = 0, den = 0;
      for (const d of allFranqs) {
        if (id === 'utilizChart')  { num += d.saldo||0; den += (d.saldo||0)+(d.cupo||0); }
        else if (id === 'moraChart')    { num += d.int_mora||0; den += d.int_corr||0; }
        else if (id === 'churnChart')   { num += d.canceladas||0; den += d.vigentes_mes||0; }
        else if (id === 'castigoChart') { num += d.castigos||0; den += d.saldo||0; }
      }
      return den > 0 ? +(num/den*100).toFixed(3) : null;
    });
    const hasData = [...vD,...mD].some(v => v != null && v > 0);
    if (!hasData) {
      const el = document.getElementById(id);
      if (el) { const ctx = el.getContext('2d'); if(ctx) ctx.clearRect(0,0,el.width,el.height); }
      if (CHARTS[id]) { CHARTS[id].destroy(); delete CHARTS[id]; }
      return;
    }
    mkChart(id, {
      type:"line",
      data:{labels, datasets:[
        {label:"Visa",       data:vD,   borderColor:VISA,     backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:2, spanGaps:true},
        {label:"Mastercard", data:mD,   borderColor:MC,       backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:2, spanGaps:true},
        {label:"Mercado",    data:totD, borderColor:"#94A3B8", backgroundColor:"transparent", fill:false, pointRadius:0, tension:0.3, borderWidth:1.5, borderDash:[5,4], spanGaps:true},
      ]},
      options:{
        responsive:true, maintainAspectRatio:false,
        layout:{padding:{top:10,right:10,bottom:2,left:2}},
        interaction:{mode:"index",intersect:false},
        plugins:{legend:{display:false},tooltip:{mode:"index",intersect:false,callbacks:{label:c=>`${c.dataset.label}: ${c.parsed.y != null ? c.parsed.y.toFixed(2)+fmtStr : "—"}`}}},
        scales:{
          x:{ticks:{maxRotation:0,autoSkip:true,maxTicksLimit:20,font:{size:9}},grid:{display:false}},
          y:{grace:"8%",ticks:{callback:v=>v.toFixed(1)+fmtStr,font:{size:9}},grid:{color:"#F2F4F7"}}
        }
      }
    });
  }

  carteraLine("utilizChart",
    d => (d.saldo+d.cupo)>0 ? d.saldo/(d.saldo+d.cupo)*100 : null, "%");
  carteraLine("moraChart",
    d => d.int_corr>0 ? d.int_mora/d.int_corr*100 : null, "%");
  carteraLine("churnChart",
    d => d.vigentes_mes>0 ? d.canceladas/d.vigentes_mes*100 : null, "%");
  carteraLine("castigoChart",
    d => d.saldo>0 ? d.castigos/d.saldo*100 : null, "%");
}

// ── Contribución por banco ───────────────────────────────────────────────────
function renderContribBancos(rows, ultimo, penult) {
  // Aggregate per-bank per-franquicia for ultimo and penult periods
  // rows are already filtered by banco; ultimo/penult come from the filtered map
  const totU = {}, totP = {};
  const franqsInRows = [...new Set(rows.map(r=>r.FRANQUICIA))].filter(f=>["VISA","MASTERCARD","AMERICAN EXPRESS","DINERS","OTRAS TARJETAS DE CREDITO"].includes(f));
  for (const r of rows) {
    if (!franqsInRows.includes(r.FRANQUICIA)) continue;
    const k = r.ENTIDAD + "|" + r.FRANQUICIA;
    // Match the period key: if ultimo is YYYY (annual), match by year; else exact month
    const rKey = ultimo.length === 4 ? r.MES.slice(0,4) : r.MES;
    if (rKey === ultimo) totU[k] = (totU[k]||0) + (r.MTO_COMPRAS_CREDITO||0);
    if (rKey === penult) totP[k] = (totP[k]||0) + (r.MTO_COMPRAS_CREDITO||0);
  }
  // Total market (all franquicias, all banks) for each period
  const mktU = Object.values(totU).reduce((s,v)=>s+v,0);
  const mktP = Object.values(totP).reduce((s,v)=>s+v,0);

  function renderForFranq(franq, elId, subId) {
    const banks = [...new Set(rows.map(r=>r.ENTIDAD))];
    const entries = banks.map(b => {
      const u = totU[b+"|"+franq]||0;
      const p = totP[b+"|"+franq]||0;
      const msU = mktU>0 ? u/mktU*100 : 0;
      const msP = mktP>0 ? p/mktP*100 : 0;
      return {bank: fmtBank(b), diff: msU-msP, curr: msU};
    }).filter(e => Math.abs(e.diff)>0.005 || e.curr>0.1);
    entries.sort((a,b) => b.diff - a.diff);
    const gainers = entries.filter(e=>e.diff>0).slice(0,5);
    const losers  = [...entries].filter(e=>e.diff<0).sort((a,b)=>a.diff-b.diff).slice(0,5);
    const top = [...gainers,...losers].filter((e,i,arr)=>arr.findIndex(x=>x.bank===e.bank)===i);
    const color = franq === "VISA" ? VISA : MC;
    let html = "";
    for (const e of top) {
      const bw = Math.min(Math.abs(e.diff)*40, 100);
      const bc = e.diff >= 0 ? color : "#D0D5DD";
      const ch = Math.abs(e.diff)<0.005
        ? `<span class="chip chip-neu">~0pp</span>`
        : e.diff>0
          ? `<span class="chip chip-up">+${e.diff.toFixed(2)}pp</span>`
          : `<span class="chip chip-down">${e.diff.toFixed(2)}pp</span>`;
      html += `<div style="margin-bottom:9px">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:3px">
          <span style="font-size:11px;color:var(--muted);max-width:60%;overflow:hidden;text-overflow:ellipsis;white-space:nowrap">${e.bank}</span>
          <span style="font-size:11px;white-space:nowrap">${ch} <strong>${e.curr.toFixed(2)}%</strong></span>
        </div>
        <div class="bar-bg"><div class="bar-fill" style="width:${bw}%;background:${bc}"></div></div>
      </div>`;
    }
    document.getElementById(elId).innerHTML = html || '<em style="color:var(--faint);font-size:11px">Sin variación en el período</em>';
    document.getElementById(subId).textContent = `Δ pp: ${ultimo} vs ${penult}`;
  }

  renderForFranq("VISA",       "contrib-banco-visa", "contrib-banco-visa-sub");
  renderForFranq("MASTERCARD", "contrib-banco-mc",   "contrib-banco-mc-sub");
}

// ── VIZ 11 — Snapshot ────────────────────────────────────────────────────────
function renderSnap(map, periods, rawMonthlyMap) {
  // ultimo = last period in filtered view
  const ultimo  = periods[periods.length-1];
  // For YoY: find the same period 12 months prior in the full monthly map
  // If filtered to a specific month, ultimo = "YYYY-MM"; prior = "YYYY-1-MM"
  // If filtered to a year (annual), ultimo = "YYYY"; prior = "YYYY-1"
  function findPrior(key, fullMap) {
    if (key.length === 4) {
      // annual: prior year
      const py = String(parseInt(key)-1);
      return fullMap[py] ? py : null;
    } else {
      // monthly: 12 months back
      const d = new Date(key+"-01");
      d.setMonth(d.getMonth()-12);
      const py = d.toISOString().slice(0,7);
      return fullMap[py] ? py : null;
    }
  }
  const priorKey = findPrior(ultimo, rawMonthlyMap) || periods[0];

  let html = "";
  for (const f of ["VISA","MASTERCARD"]) {
    const u = map[ultimo]?.[f]||{};
    // current values from filtered map
    // prior values from full monthly map (same calendar period -12m)
    const p = rawMonthlyMap[priorKey]?.[f]||{};

    const ms=(u.ms||0)*100, dMs=((u.ms||0)-(p.ms||0))*100;
    const ticket  = u.num_nal>0 ? u.mto_nal/u.num_nal : null;
    const ticketP = p.num_nal>0 ? p.mto_nal/p.num_nal : null;
    const dFact   = p.mto_tot>0 ? (u.mto_tot/p.mto_tot-1)*100 : null;
    const dTk     = ticket&&ticketP ? (ticket/ticketP-1)*100 : null;
    const dNal    = p.mto_nal>0 ? (u.mto_nal/p.mto_nal-1)*100 : null;
    const dExt    = p.mto_ext>0 ? (u.mto_ext/p.mto_ext-1)*100 : null;
    const dTxn    = (p.num_nal+p.num_ext)>0 ? ((u.num_nal+u.num_ext)/(p.num_nal+p.num_ext)-1)*100 : null;
    const dVig    = p.vigentes>0 ? (u.vigentes/p.vigentes-1)*100 : null;
    const churn   = u.vigentes_mes>0 ? u.canceladas/u.vigentes_mes*100 : null;
    const util    = (u.saldo+u.cupo)>0 ? u.saldo/(u.saldo+u.cupo)*100 : null;
    const mora    = u.int_corr>0 ? u.int_mora/u.int_corr*100 : null;
    const tasa    = u.saldo>0 ? u.castigos/u.saldo*100 : null;
    const pExt    = u.mto_tot>0 ? (u.mto_ext/u.mto_tot*100).toFixed(1)+"%" : "—";
    const chipMs  = chip(dMs,"pp");
    const yoyLabel = priorKey ? ` vs ${priorKey}` : "";
    const isDebMode = currentTipo !== "credito";
    const rows = [
      ["Facturación total",    fmtB(u.mto_tot),  chip(dFact,"%")],
      ...(!isDebMode ? [
        ["Facturación nacional",  fmtB(u.mto_nal),  chip(dNal,"%")],
        ["Facturación exterior",  fmtB(u.mto_ext),  chip(dExt,"%")],
        ["% exterior",            pExt, ""],
      ] : []),
      ["Ticket promedio",      ticket?"$"+Math.round(ticket/1000).toLocaleString("es-CO")+"K":"—", chip(dTk,"%")],
      ["N° transacciones",     Math.round((u.num_nal||0)+(u.num_ext||0)).toLocaleString("es-CO"), chip(dTxn,"%")],
      ["Tarjetas vigentes",    Math.round(u.vigentes||0).toLocaleString("es-CO"), chip(dVig,"%")],
      ["Churn rate",           churn!=null?churn.toFixed(2)+"%":"—", ""],
      ["Utilización de cupo",  util!=null?util.toFixed(1)+"%":"—", ""],
      ["Ratio de mora",        mora!=null?mora.toFixed(2)+"%":"—", ""],
      ["Tasa de castigo",      tasa!=null?tasa.toFixed(3)+"%":"—", ""],
    ];
    html += `<div class="snap-card" style="border-top:4px solid ${C[f]}">
      <div style="display:flex;align-items:center;gap:8px;margin-bottom:4px">
        <div style="width:10px;height:10px;border-radius:2px;background:${C[f]}"></div>
        <div style="font-size:15px;font-weight:700">${f}</div>
      </div>
      <div class="snap-ms" style="color:${C[f]}">${ms.toFixed(1)}%</div>
      <div style="display:flex;align-items:center;gap:6px;margin-bottom:4px">
        ${chipMs}<span class="snap-sub">market share — YoY${yoyLabel}</span>
      </div>
      <div class="snap-div"></div>
      <table class="snap-tbl">
        ${rows.map(([l,v,c])=>`<tr><td class="td-lbl">${l}</td><td>${v} ${c}</td></tr>`).join("")}
      </table>
    </div>`;
  }
  document.getElementById("snap-grid").innerHTML = html;
}
renderAll();
</script>
</body>
</html>"""

html_out = (TEMPLATE
    .replace("__DATA_JSON__",   dash_data_json)
    .replace("__BANCOS_JSON__", bancos_json)
    .replace("__ULTIMO_MES__",  ultimo_mes_str)
    .replace("__DEBITO_JSON__", debito_json)
    .replace("__TOTAL_JSON__",  total_json)
    .replace("__RETIROS_JSON__", retiros_json)
)

with open("sfc_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_out)

kb = len(html_out) / 1024
print(f"✅ sfc_dashboard.html generado ({kb:.0f} KB)")
print(f"   Período: {df_dash['MES'].min()} → {ultimo_mes_str}")
print(f"   Abre el archivo directamente en Chrome/Firefox — no requiere servidor")


## 7. Export CSVs

In [ ]:
# ── 7. Exportar todos los datasets ────────────────────────────────────────────
exports = {
    'sfc_tarjetas.csv':           df_tarjetas,
    'sfc_transacciones_credito.csv': df_tx,
    'sfc_transacciones_debito.csv':  df_debito,
    'sfc_cartera.csv':             df_cartera,
    'sfc_saldos.csv':              df_saldos,
    'sfc_tii.csv':                 df_tii,
    'sfc_master.csv':              df_master,
    'sfc_agregado_franquicia.csv': df_agg,
}

for nombre, df_exp in exports.items():
    df_exp.to_csv(nombre, index=False)
    print(f'✅ {nombre} — {len(df_exp):,} filas, {df_exp.shape[1]} columnas')

print('\n🏁 Todos los archivos exportados correctamente.')